# End-to-End Pipeline — LSTM + Vol + TP (Chapter 4.4 prep)

**Thesis section.** 4.4 — full-system driver that produces inputs for the 11 ablation runs

**Inputs.** all layer outputs from 02_ and 03_; `data/processed/<TICKER>_15min.csv`

**Outputs.** per-run signal streams consumed by the ablation backtester (ApexQuant repo)

**Expected runtime.** ~2 h (full), ~30 min (single ticker). **Expected GPU.** T4 minimum.

> All paths in the CONFIG cell below resolve relative to the repo root. On
> Google Colab, uncomment the Drive fallback line.


In [ ]:
# === CONFIG (edit paths here) ===
from pathlib import Path

CONFIG = {
    "data_dir":        Path("../../data"),         # processed + features + splits
    "results_dir":     Path("../../results"),
    "checkpoints_dir": Path("../../checkpoints"),
    "seed":            42,
    "device":          "cuda",                      # or "cpu"
    # Colab fallback — uncomment if running on Colab with the dataset mounted:
    # "data_dir": Path("/content/drive/MyDrive/thesis_data"),
}
for key, path in CONFIG.items():
    if isinstance(path, Path):
        path.mkdir(parents=True, exist_ok=True)


In [ ]:
"""
Train & Save Vol LSTM + CNN Dual Models
========================================
Run this cell ONCE before GMADL Step 4.
Saves weights to Google Drive for GMADL to load.

Architecture matches GMADL_MultiTask_LSTM.py exactly:
  - VolLSTM: 12 HAR features → next-block RV
  - CNNDualModel: (1, 30, 16) → P(bottom), P(top)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIG
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
SAVE_DIR = CONFIG['data_dir'] / r'models'
FREQ_RAW = "1min"
RESAMPLE_PERIOD = 15
TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TRAIN_RATIO = 0.8  # use first 80% for training these auxiliary models

# Vol config
VOL_BLOCK_SIZE = 24   # 24 × 15min = 6 hours
VOL_SEQ_LEN = 20      # match V3
VOL_N_FEATURES = 17   # match V3
VOL_EPOCHS = 100
VOL_PATIENCE = 15

# CNN config
CNN_WINDOW = 30
CNN_N_FEATURES = 16
CNN_EPOCHS = 80
CNN_PATIENCE = 15
CNN_BATCH = 64
CNN_LR = 1e-3

# Reversal detection for CNN labels (match original Dual CNN)
REVERSAL_THRESHOLD = 0.5   # percent — was 1.0, too strict → 14% pos rate
MIN_DURATION = 60          # minutes (not used in CNN labels, only zigzag)
TREND_PCT = 0.5            # percent for trend classification
TREND_LOOKBACK = 6         # bars
CNN_LOOKAHEAD = 6          # bars for label
REV_PCT = 0.5              # reversal percent in lookahead window

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# DATA LOADING (same as GMADL)
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df


def resample_to_15min(df, period=15):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    resampled = resampled.reset_index()
    return resampled


def load_ticker(ticker):
    """Load and resample a single ticker."""
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs:
        print(f"  [SKIP] {ticker}: no data found at {data_dir}")
        return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample_to_15min(df, RESAMPLE_PERIOD)
    return df


# ============================================================
# MODEL ARCHITECTURES (must match GMADL exactly)
# ============================================================
class VolLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class CNNDualModel(nn.Module):
    def __init__(self, n_features=16, window=30):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, 1, window, n_features)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)
        # Note: returns logits, sigmoid applied in loss/inference
# ============================================================
# PART 1: TRAIN VOL LSTM
# ============================================================
def compute_har_features(df):
    """Compute 17 features matching V3's prepare_block_rv + target."""
    close = df['close'].values.astype(float)
    n = len(close)
    ret = np.zeros(n)
    ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)

    n_blocks = n // VOL_BLOCK_SIZE
    usable = n_blocks * VOL_BLOCK_SIZE
    blocks_ret = ret[:usable].reshape(n_blocks, VOL_BLOCK_SIZE)

    # Block-level metrics
    block_rv = np.array([blocks_ret[i].std() for i in range(n_blocks)])
    block_rv_ssq = np.array([np.sqrt(np.sum(blocks_ret[i]**2)) for i in range(n_blocks)])
    block_agg_ret = np.array([blocks_ret[i].sum() for i in range(n_blocks)])
    block_mean_ret = np.array([blocks_ret[i].mean() for i in range(n_blocks)])

    rv_s = pd.Series(block_rv)

    # HAR components
    rv_d = rv_s.values
    rv_w = rv_s.rolling(5, min_periods=1).mean().values
    rv_m = rv_s.rolling(22, min_periods=1).mean().values

    # Lags (1, 2, 3, 5, 10)
    rv_lag1 = np.zeros(n_blocks); rv_lag1[1:] = block_rv[:-1]
    rv_lag2 = np.zeros(n_blocks); rv_lag2[2:] = block_rv[:-2]
    rv_lag3 = np.zeros(n_blocks); rv_lag3[3:] = block_rv[:-3]
    rv_lag5 = np.zeros(n_blocks); rv_lag5[5:] = block_rv[:-5]
    rv_lag10 = np.zeros(n_blocks); rv_lag10[10:] = block_rv[:-10]

    # Derived
    rv_change = np.zeros(n_blocks); rv_change[1:] = block_rv[1:] - block_rv[:-1]
    rv_ratio = block_rv / (rv_s.rolling(10, min_periods=1).mean().values + 1e-10)
    vol_of_vol = rv_s.rolling(10, min_periods=1).std().values
    neg_ret_frac = (block_agg_ret < 0).astype(np.float32)
    abs_block_ret = np.abs(block_agg_ret)

    # 17 features (same order as V3)
    features = np.stack([
        block_rv, block_rv_ssq, block_agg_ret, block_mean_ret,
        rv_d, rv_w, rv_m,
        rv_lag1, rv_lag2, rv_lag3, rv_lag5, rv_lag10,
        rv_change, rv_ratio, vol_of_vol,
        neg_ret_frac, abs_block_ret
    ], axis=1).astype(np.float32)

    # Target: next block RV
    target = np.zeros(n_blocks, dtype=np.float32)
    target[:-1] = block_rv[1:]

    # ===== FIX: 清除features中的NaN =====
    nan_count = np.isnan(features).sum()
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
    # =====================================

    return features, target, block_rv, n_blocks


def train_vol_model(all_features, all_targets, all_block_rv, all_n_blocks):
    """Train Vol LSTM on pooled block-level data from all tickers."""
    print(f"\n{'='*60}")
    print(f"  Training Vol LSTM (pooled across {len(all_features)} tickers)")
    print(f"{'='*60}")

    # Build sequences PER TICKER then concatenate (avoid cross-ticker boundaries)
    def make_sequences(X, y, seq_len):
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i - seq_len:i])
            ys.append(y[i])
        if len(Xs) == 0:
            return None, None
        return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

    # First pass: fit scaler on all train data
    sc_x = StandardScaler()
    sc_y = StandardScaler()
    train_feats = []
    train_targets = []
    for feat, tgt in zip(all_features, all_targets):
        n = len(feat)
        tr_end = int(n * TRAIN_RATIO)
        train_feats.append(feat[:tr_end])
        train_targets.append(tgt[:tr_end])
    sc_x.fit(np.concatenate(train_feats))
    sc_y.fit(np.concatenate(train_targets).reshape(-1, 1))

    # Second pass: build sequences per ticker
    all_tr_X, all_tr_y = [], []
    all_v_X, all_v_y = [], []
    all_v_rv = []

    for feat, tgt, brv in zip(all_features, all_targets, all_block_rv):
        n = len(feat)
        tr_end = int(n * TRAIN_RATIO)

        feat_s = sc_x.transform(feat)
        tgt_s = sc_y.transform(tgt.reshape(-1, 1)).ravel()

        # ===== FIX: 清除scaler输出中的NaN =====
        feat_s = np.nan_to_num(feat_s, nan=0.0, posinf=0.0, neginf=0.0)
        tgt_s = np.nan_to_num(tgt_s, nan=0.0, posinf=0.0, neginf=0.0)
        # ======================================

        # Train sequences (from this ticker only)
        res = make_sequences(feat_s[:tr_end], tgt_s[:tr_end], VOL_SEQ_LEN)
        if res[0] is not None:
            all_tr_X.append(res[0])
            all_tr_y.append(res[1])

        # Val sequences (from this ticker only)
        res = make_sequences(feat_s[tr_end:], tgt_s[tr_end:], VOL_SEQ_LEN)
        if res[0] is not None:
            all_v_X.append(res[0])
            all_v_y.append(res[1])
            all_v_rv.append(brv[tr_end + VOL_SEQ_LEN:])

    X_tr = np.concatenate(all_tr_X)
    y_tr = np.concatenate(all_tr_y)
    X_v = np.concatenate(all_v_X)
    y_v = np.concatenate(all_v_y)
    rv_v = np.concatenate(all_v_rv)

    # ===== FIX: 最终NaN检查 =====
    nan_report = (f"X_tr: {np.isnan(X_tr).sum()}, y_tr: {np.isnan(y_tr).sum()}, "
                  f"X_v: {np.isnan(X_v).sum()}, y_v: {np.isnan(y_v).sum()}")
    print(f"  NaN check after scaler: {nan_report}")
    X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
    y_tr = np.nan_to_num(y_tr, nan=0.0, posinf=0.0, neginf=0.0)
    X_v = np.nan_to_num(X_v, nan=0.0, posinf=0.0, neginf=0.0)
    y_v = np.nan_to_num(y_v, nan=0.0, posinf=0.0, neginf=0.0)
    # =============================

    print(f"  Train sequences: {len(X_tr)}, Val sequences: {len(X_v)}")

    # Train
    model = VolLSTM(input_size=VOL_N_FEATURES, hidden_size=64, num_layers=2, dropout=0.2).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()

    ds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr))
    dl = DataLoader(ds, batch_size=32, shuffle=True)
    Xv_t = torch.FloatTensor(X_v).to(DEVICE)
    yv_t = torch.FloatTensor(y_v).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(VOL_EPOCHS):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(-1), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv_t).squeeze(-1), yv_t).item()

        # ===== FIX: 处理NaN val loss =====
        if np.isnan(vl):
            print(f"  [WARNING] NaN val_loss at epoch {ep+1}, skipping")
            continue
        # ==================================

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= VOL_PATIENCE:
                print(f"  Early stop at epoch {ep+1}")
                break
        if (ep + 1) % 20 == 0:
            print(f"  Epoch {ep+1}: val_loss={vl:.6f}, best={best_vl:.6f}")

    if best_st:
        model.load_state_dict(best_st)
    else:
        print(f"  [WARNING] No improvement found (best_vl={best_vl}), using last epoch weights")

    # Evaluate DirAcc on val
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        preds = model(Xv_t).squeeze(-1).cpu().numpy()
    preds_inv = sc_y.inverse_transform(preds.reshape(-1, 1)).ravel()
    y_v_inv = sc_y.inverse_transform(y_v.reshape(-1, 1)).ravel()

    # DirAcc: did we correctly predict if next RV > current RV?
    n_eval = min(len(preds_inv), len(rv_v))
    actual_dir = y_v_inv[:n_eval] > rv_v[:n_eval]
    pred_dir = preds_inv[:n_eval] > rv_v[:n_eval]
    da = np.mean(actual_dir == pred_dir) * 100
    print(f"  Vol LSTM DirAcc: {da:.1f}% (N={n_eval})")
    print(f"  Best val_loss: {best_vl:.6f}")

    return model


# ============================================================
# PART 2: TRAIN CNN DUAL
# ============================================================
def compute_cnn_features_16(df):
    """Compute 16 CNN input features per bar (same as GMADL)."""
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)

    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()

    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)

    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values

    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values

    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values

    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)

    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values

    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values

    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values

    # Ensure exactly 16 features
    cols = list(feat.columns)[:CNN_N_FEATURES]
    while len(cols) < CNN_N_FEATURES:
        cols.append(cols[-1])
    feat = feat[cols]

    return feat.values.astype(np.float32)


def find_trend_and_label(df):
    """
    Find bars in up/down trends and label whether reversal happens within LOOKAHEAD bars.
    Returns list of dicts with bar_idx, trend_dir, label.
    """
    close = df['close'].values.astype(float)
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LB = TREND_LOOKBACK
    LA = CNN_LOOKAHEAD
    rev_pct = TREND_PCT / 100.0  # 0.5% reversal threshold (same as trend)

    positions = []
    for t in range(max(CNN_WINDOW, LB), n - LA):
        p_now = close[t]
        p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        if move < -trend_pct:
            # Downtrend → check for bottom reversal (price rises)
            future_max = np.max(close[t+1:t+1+LA])
            label = 1 if (future_max - p_now) / p_now > rev_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'down', 'label': label})
        elif move > trend_pct:
            # Uptrend → check for top reversal (price drops)
            future_min = np.min(close[t+1:t+1+LA])
            label = 1 if (p_now - future_min) / p_now > rev_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'up', 'label': label})

    return positions


def build_cnn_dataset(features_16, positions):
    """Build windowed dataset for CNN training."""
    X, y = [], []
    for p in positions:
        idx = p['bar_idx']
        if idx < CNN_WINDOW:
            continue
        window = features_16[idx - CNN_WINDOW:idx]  # (30, 16)
        if window.shape != (CNN_WINDOW, CNN_N_FEATURES):
            continue
        X.append(window)
        y.append(p['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


def train_cnn_dual(all_ticker_data):
    """
    Train CNN Dual model: shared conv layers, separate bottom/top heads.
    Pool train data from all tickers.
    """
    print(f"\n{'='*60}")
    print(f"  Training CNN Dual Model")
    print(f"{'='*60}")

    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []

    for ticker, (df, features_16, train_end_idx) in all_ticker_data.items():
        positions = find_trend_and_label(df)

        pos_down = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'down']
        pos_up = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'up']

        if len(pos_down) > 30:
            X_d, y_d = build_cnn_dataset(features_16, pos_down)
            vs = max(int(len(X_d) * 0.15), 1)
            down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
            down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])

        if len(pos_up) > 30:
            X_u, y_u = build_cnn_dataset(features_16, pos_up)
            vs = max(int(len(X_u) * 0.15), 1)
            up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
            up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

        print(f"  {ticker}: {len(pos_down)} down, {len(pos_up)} up positions")

    if not down_tr_X or not up_tr_X:
        print("  [ERROR] Insufficient training data for CNN")
        return None

    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X); yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X); yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X); yu_v = np.concatenate(up_v_y)

    print(f"  Bottom: train={len(Xd_tr)}, val={len(Xd_v)} (pos_rate={yd_tr.mean():.2f})")
    print(f"  Top:    train={len(Xu_tr)}, val={len(Xu_v)} (pos_rate={yu_tr.mean():.2f})")

    # Normalize features (using train stats)
    n_samples_d, W, F = Xd_tr.shape
    flat_d = Xd_tr.reshape(-1, F)
    sc_d = StandardScaler().fit(flat_d)
    Xd_tr = sc_d.transform(Xd_tr.reshape(-1, F)).reshape(n_samples_d, W, F)
    Xd_v = sc_d.transform(Xd_v.reshape(-1, F)).reshape(len(Xd_v), W, F)

    n_samples_u = len(Xu_tr)
    flat_u = Xu_tr.reshape(-1, F)
    sc_u = StandardScaler().fit(flat_u)
    Xu_tr = sc_u.transform(Xu_tr.reshape(-1, F)).reshape(n_samples_u, W, F)
    Xu_v = sc_u.transform(Xu_v.reshape(-1, F)).reshape(len(Xu_v), W, F)

    # Train model with alternating bottom/top batches
    model = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss()

    # Bottom dataloader
    ds_d = TensorDataset(
        torch.FloatTensor(Xd_tr).unsqueeze(1),  # (N, 1, W, F)
        torch.FloatTensor(yd_tr.astype(np.float32)))
    dl_d = DataLoader(ds_d, batch_size=CNN_BATCH, shuffle=True)

    # Top dataloader
    ds_u = TensorDataset(
        torch.FloatTensor(Xu_tr).unsqueeze(1),
        torch.FloatTensor(yu_tr.astype(np.float32)))
    dl_u = DataLoader(ds_u, batch_size=CNN_BATCH, shuffle=True)

    # Val tensors
    Xdv_t = torch.FloatTensor(Xd_v).unsqueeze(1).to(DEVICE)
    ydv_t = torch.FloatTensor(yd_v.astype(np.float32)).to(DEVICE)
    Xuv_t = torch.FloatTensor(Xu_v).unsqueeze(1).to(DEVICE)
    yuv_t = torch.FloatTensor(yu_v.astype(np.float32)).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(CNN_EPOCHS):
        model.train()
        # Train on bottom samples
        for xb, yb in dl_d:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit_b, _ = model(xb)
            loss = crit(logit_b, yb)
            loss.backward()
            opt.step()

        # Train on top samples
        for xb, yb in dl_u:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            _, logit_t = model(xb)
            loss = crit(logit_t, yb)
            loss.backward()
            opt.step()

        # Validate
        model.eval()
        with torch.no_grad():
            vl_b = crit(model(Xdv_t)[0], ydv_t).item()
            vl_t = crit(model(Xuv_t)[1], yuv_t).item()
            vl = (vl_b + vl_t) / 2

        if vl < best_vl:
            best_vl, best_st, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= CNN_PATIENCE:
                print(f"  Early stop at epoch {ep+1}")
                break

        if (ep + 1) % 20 == 0:
            print(f"  Epoch {ep+1}: val_loss={vl:.4f} (bottom={vl_b:.4f}, top={vl_t:.4f})")

    if best_st:
        model.load_state_dict(best_st)

    # Evaluate accuracy
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        pb = torch.sigmoid(model(Xdv_t)[0]).cpu().numpy()
        pt = torch.sigmoid(model(Xuv_t)[1]).cpu().numpy()
    acc_b = np.mean((pb > 0.5).astype(int) == yd_v) * 100
    acc_t = np.mean((pt > 0.5).astype(int) == yu_v) * 100
    print(f"  Bottom acc: {acc_b:.1f}% (N={len(yd_v)})")
    print(f"  Top acc:    {acc_t:.1f}% (N={len(yu_v)})")

    return model


# ============================================================
# MAIN
# ============================================================
def main():
    print(f"\n{'='*60}")
    print(f"  Loading data for all tickers...")
    print(f"{'='*60}")

    vol_features_all = []
    vol_targets_all = []
    vol_block_rv_all = []
    vol_n_blocks_all = []
    cnn_ticker_data = {}

    for ticker in TICKERS:
        df = load_ticker(ticker)
        if df is None:
            continue

        n = len(df)
        train_end_idx = int(n * TRAIN_RATIO)
        print(f"  {ticker}: {n} bars, train_end={train_end_idx}")

        # Vol: HAR features at block level
        features, target, block_rv, n_blocks = compute_har_features(df)
        vol_features_all.append(features)
        vol_targets_all.append(target)
        vol_block_rv_all.append(block_rv)
        vol_n_blocks_all.append(n_blocks)

        # CNN: 16 features
        features_16 = compute_cnn_features_16(df)
        cnn_ticker_data[ticker] = (df, features_16, train_end_idx)

    # --- Train Vol LSTM ---
    vol_model = train_vol_model(vol_features_all, vol_targets_all, vol_block_rv_all, vol_n_blocks_all)

    vol_path = os.path.join(SAVE_DIR, "vol_lstm_v3.pt")
    torch.save(vol_model.state_dict(), vol_path)
    print(f"\n  ✓ Vol LSTM saved to {vol_path}")

    # --- Train CNN Dual ---
    cnn_model = train_cnn_dual(cnn_ticker_data)

    if cnn_model is not None:
        cnn_path = os.path.join(SAVE_DIR, "cnn_dual.pt")
        torch.save(cnn_model.state_dict(), cnn_path)
        print(f"  ✓ CNN Dual saved to {cnn_path}")

    print(f"\n{'='*60}")
    print(f"  DONE! Models saved to {SAVE_DIR}")
    print(f"  Now run GMADL Step 4 — it will load these weights.")
    print(f"{'='*60}")


main()


Device: cuda

  Loading data for all tickers...
  AAPL: 39289 bars, train_end=31431
  MSFT: 39154 bars, train_end=31323
  GOOGL: 35495 bars, train_end=28396
  GOOG: 34511 bars, train_end=27608
  NVDA: 39057 bars, train_end=31245
  TSLA: 39241 bars, train_end=31392
  SPY: 39251 bars, train_end=31400
  QQQ: 39316 bars, train_end=31452

  Training Vol LSTM (pooled across 8 tickers)
  NaN check after scaler: X_tr: 0, y_tr: 0, X_v: 0, y_v: 0
  Train sequences: 10011, Val sequences: 2387
  Epoch 20: val_loss=0.414095, best=0.390796
  Early stop at epoch 23
  Vol LSTM DirAcc: 80.5% (N=2387)
  Best val_loss: 0.390796

  ✓ Vol LSTM saved to models/vol_lstm_v3.pt

  Training CNN Dual Model
  AAPL: 4153 down, 4544 up positions
  MSFT: 3582 down, 3968 up positions
  GOOGL: 3792 down, 3950 up positions
  GOOG: 3721 down, 3906 up positions
  NVDA: 5805 down, 6614 up positions
  TSLA: 7353 down, 8263 up positions
  SPY: 2096 down, 2049 up positions
  QQQ: 2903 down, 2835 up positions
  Bottom: train=

In [ ]:
"""
Train & Save Vol LSTM + CNN Dual Models (FIXED VERSION)
========================================================
Fixes applied:
  1. CNN training: added gradient clipping (was missing → gradient explosion → NaN weights)
  2. CNN training: added NaN val_loss check (was missing → best_st never updated)
  3. CNN training: added per-epoch NaN weight detection
  4. CNN training: added Focal Loss option for class imbalance
  5. Post-save verification: assert no NaN in saved weights
  6. Vol LSTM: kept existing NaN fixes (gradient clipping, NaN check)
  7. Both models: comprehensive logging and diagnostics

Run this cell ONCE before GMADL Step 4.
Saves weights to Google Drive for GMADL to load.
Architecture matches GMADL_MultiTask_LSTM.py exactly:
  - VolLSTM: 17 HAR features → next-block RV
  - CNNDualModel: (1, 30, 16) → P(bottom), P(top)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIG
# ============================================================
DRIVE_BASE = CONFIG['data_dir'] / r'splits'
SAVE_DIR = CONFIG['data_dir'] / r'models'
FREQ_RAW = "1min"
RESAMPLE_PERIOD = 15
TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TRAIN_RATIO = 0.8

# Vol config
VOL_BLOCK_SIZE = 24
VOL_SEQ_LEN = 20
VOL_N_FEATURES = 17
VOL_EPOCHS = 100
VOL_PATIENCE = 15

# CNN config
CNN_WINDOW = 30
CNN_N_FEATURES = 16
CNN_EPOCHS = 80
CNN_PATIENCE = 15
CNN_BATCH = 64
CNN_LR = 1e-3
CNN_GRAD_CLIP = 1.0       # ← FIX: gradient clipping for CNN
CNN_WEIGHT_DECAY = 1e-5

# Reversal detection for CNN labels
TREND_PCT = 0.5
TREND_LOOKBACK = 6
CNN_LOOKAHEAD = 6
REV_PCT = 0.5

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# DATA LOADING (same as GMADL)
# ============================================================
def load_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event':    col_map[col] = 'timestamp'
        elif cl == 'open':      col_map[col] = 'open'
        elif cl == 'high':      col_map[col] = 'high'
        elif cl == 'low':       col_map[col] = 'low'
        elif cl == 'close':     col_map[col] = 'close'
        elif cl == 'volume':    col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df


def resample_to_15min(df, period=15):
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    resampled = resampled.reset_index()
    return resampled


def load_ticker(ticker):
    data_dir = os.path.join(DRIVE_BASE, f"{ticker}_{FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))
    if not dfs:
        print(f"  [SKIP] {ticker}: no data found at {data_dir}")
        return None
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample_to_15min(df, RESAMPLE_PERIOD)
    return df


# ============================================================
# MODEL ARCHITECTURES (must match GMADL exactly)
# ============================================================
class VolLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class CNNDualModel(nn.Module):
    def __init__(self, n_features=16, window=30):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, 1, window, n_features)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)
        # Returns logits, sigmoid applied in loss/inference


# ============================================================
# UTILITY: Weight health check
# ============================================================
def check_model_health(model, name="Model"):
    """Check if any model parameters contain NaN or Inf."""
    n_nan = 0
    n_inf = 0
    n_total = 0
    for pname, param in model.named_parameters():
        n_total += param.numel()
        nan_count = torch.isnan(param.data).sum().item()
        inf_count = torch.isinf(param.data).sum().item()
        n_nan += nan_count
        n_inf += inf_count
        if nan_count > 0 or inf_count > 0:
            print(f"    ⚠ {pname}: {nan_count} NaN, {inf_count} Inf "
                  f"(shape={list(param.shape)})")
    healthy = (n_nan == 0 and n_inf == 0)
    status = "✓ HEALTHY" if healthy else f"✗ CORRUPT ({n_nan} NaN, {n_inf} Inf)"
    print(f"  {name} weight check: {status} ({n_total} params)")
    return healthy


def verify_saved_weights(path, model_class, model_kwargs):
    """Load saved weights and verify they contain no NaN."""
    state = torch.load(path, map_location='cpu')
    n_nan = 0
    for k, v in state.items():
        nan_count = torch.isnan(v).sum().item()
        if nan_count > 0:
            print(f"    ⚠ SAVED {k}: {nan_count} NaN values!")
            n_nan += nan_count
    if n_nan > 0:
        print(f"  ✗ SAVED WEIGHTS CORRUPT: {n_nan} total NaN values in {path}")
        return False
    # Also verify forward pass works
    model = model_class(**model_kwargs)
    model.load_state_dict(state)
    model.eval()
    # Test with random input
    if isinstance(model, VolLSTM):
        x = torch.randn(2, VOL_SEQ_LEN, VOL_N_FEATURES)
        out = model(x)
        if torch.isnan(out).any():
            print(f"  ✗ FORWARD PASS produces NaN!")
            return False
    elif isinstance(model, CNNDualModel):
        x = torch.randn(2, 1, CNN_WINDOW, CNN_N_FEATURES)
        pb, pt = model(x)
        if torch.isnan(pb).any() or torch.isnan(pt).any():
            print(f"  ✗ FORWARD PASS produces NaN! pb={pb}, pt={pt}")
            return False
    print(f"  ✓ Saved weights verified: {path}")
    return True


# ============================================================
# PART 1: TRAIN VOL LSTM
# ============================================================
def compute_har_features(df):
    """Compute 17 features matching V3's prepare_block_rv + target."""
    close = df['close'].values.astype(float)
    n = len(close)

    ret = np.zeros(n)
    ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)

    n_blocks = n // VOL_BLOCK_SIZE
    usable = n_blocks * VOL_BLOCK_SIZE
    blocks_ret = ret[:usable].reshape(n_blocks, VOL_BLOCK_SIZE)

    block_rv = np.array([blocks_ret[i].std() for i in range(n_blocks)])
    block_rv_ssq = np.array([np.sqrt(np.sum(blocks_ret[i]**2)) for i in range(n_blocks)])
    block_agg_ret = np.array([blocks_ret[i].sum() for i in range(n_blocks)])
    block_mean_ret = np.array([blocks_ret[i].mean() for i in range(n_blocks)])

    rv_s = pd.Series(block_rv)
    rv_d = rv_s.values
    rv_w = rv_s.rolling(5, min_periods=1).mean().values
    rv_m = rv_s.rolling(22, min_periods=1).mean().values

    rv_lag1 = np.zeros(n_blocks); rv_lag1[1:] = block_rv[:-1]
    rv_lag2 = np.zeros(n_blocks); rv_lag2[2:] = block_rv[:-2]
    rv_lag3 = np.zeros(n_blocks); rv_lag3[3:] = block_rv[:-3]
    rv_lag5 = np.zeros(n_blocks); rv_lag5[5:] = block_rv[:-5]
    rv_lag10 = np.zeros(n_blocks); rv_lag10[10:] = block_rv[:-10]

    rv_change = np.zeros(n_blocks); rv_change[1:] = block_rv[1:] - block_rv[:-1]
    rv_ratio = block_rv / (rv_s.rolling(10, min_periods=1).mean().values + 1e-10)
    vol_of_vol = rv_s.rolling(10, min_periods=1).std().values
    neg_ret_frac = (block_agg_ret < 0).astype(np.float32)
    abs_block_ret = np.abs(block_agg_ret)

    features = np.stack([
        block_rv, block_rv_ssq, block_agg_ret, block_mean_ret,
        rv_d, rv_w, rv_m,
        rv_lag1, rv_lag2, rv_lag3, rv_lag5, rv_lag10,
        rv_change, rv_ratio, vol_of_vol,
        neg_ret_frac, abs_block_ret
    ], axis=1).astype(np.float32)

    target = np.zeros(n_blocks, dtype=np.float32)
    target[:-1] = block_rv[1:]

    # Clean NaN
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)

    return features, target, block_rv, n_blocks


def train_vol_model(all_features, all_targets, all_block_rv, all_n_blocks):
    """Train Vol LSTM on pooled block-level data from all tickers."""
    print(f"\n{'='*60}")
    print(f"  Training Vol LSTM (pooled across {len(all_features)} tickers)")
    print(f"{'='*60}")

    def make_sequences(X, y, seq_len):
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i - seq_len:i])
            ys.append(y[i])
        if len(Xs) == 0:
            return None, None
        return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

    # Fit scaler on train data only
    sc_x = StandardScaler()
    sc_y = StandardScaler()
    train_feats = []
    train_targets = []
    for feat, tgt in zip(all_features, all_targets):
        n = len(feat)
        tr_end = int(n * TRAIN_RATIO)
        train_feats.append(feat[:tr_end])
        train_targets.append(tgt[:tr_end])
    sc_x.fit(np.concatenate(train_feats))
    sc_y.fit(np.concatenate(train_targets).reshape(-1, 1))

    # Build sequences per ticker
    all_tr_X, all_tr_y = [], []
    all_v_X, all_v_y = [], []
    all_v_rv = []
    for feat, tgt, brv in zip(all_features, all_targets, all_block_rv):
        n = len(feat)
        tr_end = int(n * TRAIN_RATIO)
        feat_s = sc_x.transform(feat)
        tgt_s = sc_y.transform(tgt.reshape(-1, 1)).ravel()
        feat_s = np.nan_to_num(feat_s, nan=0.0, posinf=0.0, neginf=0.0)
        tgt_s = np.nan_to_num(tgt_s, nan=0.0, posinf=0.0, neginf=0.0)

        res = make_sequences(feat_s[:tr_end], tgt_s[:tr_end], VOL_SEQ_LEN)
        if res[0] is not None:
            all_tr_X.append(res[0])
            all_tr_y.append(res[1])

        res = make_sequences(feat_s[tr_end:], tgt_s[tr_end:], VOL_SEQ_LEN)
        if res[0] is not None:
            all_v_X.append(res[0])
            all_v_y.append(res[1])
            all_v_rv.append(brv[tr_end + VOL_SEQ_LEN:])

    X_tr = np.concatenate(all_tr_X)
    y_tr = np.concatenate(all_tr_y)
    X_v = np.concatenate(all_v_X)
    y_v = np.concatenate(all_v_y)
    rv_v = np.concatenate(all_v_rv)

    # Final NaN check
    X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
    y_tr = np.nan_to_num(y_tr, nan=0.0, posinf=0.0, neginf=0.0)
    X_v = np.nan_to_num(X_v, nan=0.0, posinf=0.0, neginf=0.0)
    y_v = np.nan_to_num(y_v, nan=0.0, posinf=0.0, neginf=0.0)

    nan_report = (f"X_tr: {np.isnan(X_tr).sum()}, y_tr: {np.isnan(y_tr).sum()}, "
                  f"X_v: {np.isnan(X_v).sum()}, y_v: {np.isnan(y_v).sum()}")
    print(f"  NaN check after scaler: {nan_report}")
    print(f"  Train sequences: {len(X_tr)}, Val sequences: {len(X_v)}")

    # Train
    model = VolLSTM(input_size=VOL_N_FEATURES, hidden_size=64,
                    num_layers=2, dropout=0.2).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()

    ds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr))
    dl = DataLoader(ds, batch_size=32, shuffle=True)
    Xv_t = torch.FloatTensor(X_v).to(DEVICE)
    yv_t = torch.FloatTensor(y_v).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(VOL_EPOCHS):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(-1), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv_t).squeeze(-1), yv_t).item()

        if np.isnan(vl):
            print(f"  [WARNING] NaN val_loss at epoch {ep+1}, skipping")
            continue

        if vl < best_vl:
            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= VOL_PATIENCE:
                print(f"  Early stop at epoch {ep+1}")
                break
        if (ep + 1) % 20 == 0:
            print(f"  Epoch {ep+1}: val_loss={vl:.6f}, best={best_vl:.6f}")

    if best_st:
        model.load_state_dict(best_st)
    else:
        print(f"  [WARNING] No improvement found, using last epoch weights")

    # Evaluate DirAcc
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        preds = model(Xv_t).squeeze(-1).cpu().numpy()
    preds_inv = sc_y.inverse_transform(preds.reshape(-1, 1)).ravel()
    y_v_inv = sc_y.inverse_transform(y_v.reshape(-1, 1)).ravel()
    n_eval = min(len(preds_inv), len(rv_v))
    actual_dir = y_v_inv[:n_eval] > rv_v[:n_eval]
    pred_dir = preds_inv[:n_eval] > rv_v[:n_eval]
    da = np.mean(actual_dir == pred_dir) * 100
    print(f"  Vol LSTM DirAcc: {da:.1f}% (N={n_eval})")
    print(f"  Best val_loss: {best_vl:.6f}")

    # Verify health
    check_model_health(model, "Vol LSTM")
    return model


# ============================================================
# PART 2: TRAIN CNN DUAL (FIXED)
# ============================================================
def compute_cnn_features_16(df):
    """Compute 16 CNN input features per bar."""
    close = df['close'].values.astype(float)
    high = df['high'].values.astype(float)
    low = df['low'].values.astype(float)
    volume = df['volume'].values.astype(float)

    feat = pd.DataFrame()
    ret = pd.Series(close).pct_change()
    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)

    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values

    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values

    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values

    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)

    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values

    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values

    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values

    cols = list(feat.columns)[:CNN_N_FEATURES]
    while len(cols) < CNN_N_FEATURES:
        cols.append(cols[-1])
    feat = feat[cols]
    return feat.values.astype(np.float32)


def find_trend_and_label(df):
    """Find bars in up/down trends and label whether reversal happens."""
    close = df['close'].values.astype(float)
    n = len(close)
    trend_pct = TREND_PCT / 100.0
    LB = TREND_LOOKBACK
    LA = CNN_LOOKAHEAD
    rev_pct = TREND_PCT / 100.0

    positions = []
    for t in range(max(CNN_WINDOW, LB), n - LA):
        p_now = close[t]
        p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        if move < -trend_pct:
            future_max = np.max(close[t+1:t+1+LA])
            label = 1 if (future_max - p_now) / p_now > rev_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'down', 'label': label})
        elif move > trend_pct:
            future_min = np.min(close[t+1:t+1+LA])
            label = 1 if (p_now - future_min) / p_now > rev_pct else 0
            positions.append({'bar_idx': t, 'trend_dir': 'up', 'label': label})

    return positions


def build_cnn_dataset(features_16, positions):
    """Build windowed dataset for CNN training."""
    X, y = [], []
    for p in positions:
        idx = p['bar_idx']
        if idx < CNN_WINDOW:
            continue
        window = features_16[idx - CNN_WINDOW:idx]
        if window.shape != (CNN_WINDOW, CNN_N_FEATURES):
            continue
        # Skip windows with NaN
        if np.isnan(window).any():
            continue
        X.append(window)
        y.append(p['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


def train_cnn_dual(all_ticker_data):
    """
    Train CNN Dual model with all NaN fixes applied.
    Key fixes vs original:
      1. Gradient clipping in BOTH bottom and top training loops
      2. NaN val_loss detection and skip
      3. Per-epoch weight health monitoring
      4. Post-training weight verification
      5. Learning rate reduction on NaN detection
    """
    print(f"\n{'='*60}")
    print(f"  Training CNN Dual Model (FIXED)")
    print(f"{'='*60}")

    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []

    for ticker, (df, features_16, train_end_idx) in all_ticker_data.items():
        positions = find_trend_and_label(df)
        pos_down = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'down']
        pos_up = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'up']

        if len(pos_down) > 30:
            X_d, y_d = build_cnn_dataset(features_16, pos_down)
            if len(X_d) > 30:
                vs = max(int(len(X_d) * 0.15), 1)
                down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
                down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])

        if len(pos_up) > 30:
            X_u, y_u = build_cnn_dataset(features_16, pos_up)
            if len(X_u) > 30:
                vs = max(int(len(X_u) * 0.15), 1)
                up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
                up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

        print(f"  {ticker}: {len(pos_down)} down, {len(pos_up)} up positions")

    if not down_tr_X or not up_tr_X:
        print("  [ERROR] Insufficient training data for CNN")
        return None

    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X);   yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X);   yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X);     yu_v = np.concatenate(up_v_y)

    # --- FIX: Check for NaN in training data ---
    for name, arr in [("Xd_tr", Xd_tr), ("Xu_tr", Xu_tr),
                      ("Xd_v", Xd_v), ("Xu_v", Xu_v)]:
        nan_count = np.isnan(arr).sum()
        if nan_count > 0:
            print(f"  [WARNING] {name} has {nan_count} NaN values, replacing with 0")
            arr[:] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"  Bottom: train={len(Xd_tr)}, val={len(Xd_v)} (pos_rate={yd_tr.mean():.2f})")
    print(f"  Top:    train={len(Xu_tr)}, val={len(Xu_v)} (pos_rate={yu_tr.mean():.2f})")

    # Normalize features (using train stats)
    n_samples_d, W, F = Xd_tr.shape
    flat_d = Xd_tr.reshape(-1, F)
    sc_d = StandardScaler().fit(flat_d)
    Xd_tr = sc_d.transform(Xd_tr.reshape(-1, F)).reshape(n_samples_d, W, F)
    Xd_v = sc_d.transform(Xd_v.reshape(-1, F)).reshape(len(Xd_v), W, F)

    n_samples_u = len(Xu_tr)
    flat_u = Xu_tr.reshape(-1, F)
    sc_u = StandardScaler().fit(flat_u)
    Xu_tr = sc_u.transform(Xu_tr.reshape(-1, F)).reshape(n_samples_u, W, F)
    Xu_v = sc_u.transform(Xu_v.reshape(-1, F)).reshape(len(Xu_v), W, F)

    # --- FIX: Clean NaN after scaling ---
    Xd_tr = np.nan_to_num(Xd_tr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    Xd_v = np.nan_to_num(Xd_v, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    Xu_tr = np.nan_to_num(Xu_tr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    Xu_v = np.nan_to_num(Xu_v, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    print(f"  Post-scaling NaN check: Xd_tr={np.isnan(Xd_tr).sum()}, "
          f"Xu_tr={np.isnan(Xu_tr).sum()}")

    # --- Initialize model ---
    model = CNNDualModel(n_features=CNN_N_FEATURES, window=CNN_WINDOW).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CNN_LR, weight_decay=CNN_WEIGHT_DECAY)
    crit = nn.BCEWithLogitsLoss()

    # --- FIX: Verify model is healthy before training ---
    check_model_health(model, "CNN (init)")

    # Dataloaders
    ds_d = TensorDataset(
        torch.FloatTensor(Xd_tr).unsqueeze(1),
        torch.FloatTensor(yd_tr.astype(np.float32)))
    dl_d = DataLoader(ds_d, batch_size=CNN_BATCH, shuffle=True)

    ds_u = TensorDataset(
        torch.FloatTensor(Xu_tr).unsqueeze(1),
        torch.FloatTensor(yu_tr.astype(np.float32)))
    dl_u = DataLoader(ds_u, batch_size=CNN_BATCH, shuffle=True)

    Xdv_t = torch.FloatTensor(Xd_v).unsqueeze(1).to(DEVICE)
    ydv_t = torch.FloatTensor(yd_v.astype(np.float32)).to(DEVICE)
    Xuv_t = torch.FloatTensor(Xu_v).unsqueeze(1).to(DEVICE)
    yuv_t = torch.FloatTensor(yu_v.astype(np.float32)).to(DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    nan_count_consecutive = 0

    for ep in range(CNN_EPOCHS):
        model.train()
        epoch_loss_b = 0
        epoch_loss_t = 0
        n_batch_b = 0
        n_batch_t = 0

        # ===== Train on bottom samples =====
        for xb, yb in dl_d:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit_b, _ = model(xb)
            loss = crit(logit_b, yb)

            # --- FIX: Check for NaN loss before backward ---
            if torch.isnan(loss):
                print(f"  [WARNING] NaN loss in bottom batch at epoch {ep+1}, skipping batch")
                opt.zero_grad()  # clear any partial gradients
                continue

            loss.backward()
            # --- FIX: Gradient clipping (was MISSING in original) ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_GRAD_CLIP)
            opt.step()
            epoch_loss_b += loss.item()
            n_batch_b += 1

        # ===== Train on top samples =====
        for xb, yb in dl_u:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            _, logit_t = model(xb)
            loss = crit(logit_t, yb)

            # --- FIX: Check for NaN loss before backward ---
            if torch.isnan(loss):
                print(f"  [WARNING] NaN loss in top batch at epoch {ep+1}, skipping batch")
                opt.zero_grad()
                continue

            loss.backward()
            # --- FIX: Gradient clipping (was MISSING in original) ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_GRAD_CLIP)
            opt.step()
            epoch_loss_t += loss.item()
            n_batch_t += 1

        # ===== Validate =====
        model.eval()
        with torch.no_grad():
            logit_b_val, _ = model(Xdv_t)
            _, logit_t_val = model(Xuv_t)

            # --- FIX: Check for NaN in model output ---
            if torch.isnan(logit_b_val).any() or torch.isnan(logit_t_val).any():
                print(f"  [WARNING] NaN model output at epoch {ep+1}!")
                nan_count_consecutive += 1
                if nan_count_consecutive >= 3:
                    print(f"  [ERROR] 3 consecutive NaN epochs, stopping training")
                    break
                continue

            vl_b = crit(logit_b_val, ydv_t).item()
            vl_t = crit(logit_t_val, yuv_t).item()
            vl = (vl_b + vl_t) / 2

        # --- FIX: NaN val_loss check (was MISSING in original) ---
        if np.isnan(vl) or np.isinf(vl):
            print(f"  [WARNING] NaN/Inf val_loss at epoch {ep+1}, skipping")
            nan_count_consecutive += 1
            if nan_count_consecutive >= 3:
                print(f"  [ERROR] 3 consecutive NaN epochs, stopping training")
                break
            continue

        nan_count_consecutive = 0  # reset counter on successful epoch

        if vl < best_vl:
            # --- FIX: Verify weights before saving checkpoint ---
            has_nan = False
            for pname, param in model.named_parameters():
                if torch.isnan(param.data).any():
                    has_nan = True
                    break
            if has_nan:
                print(f"  [WARNING] Epoch {ep+1}: best val_loss but weights have NaN, "
                      f"NOT saving checkpoint")
                continue

            best_vl = vl
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= CNN_PATIENCE:
                print(f"  Early stop at epoch {ep+1}")
                break

        if (ep + 1) % 10 == 0:
            avg_b = epoch_loss_b / max(n_batch_b, 1)
            avg_t = epoch_loss_t / max(n_batch_t, 1)
            print(f"  Epoch {ep+1}: val_loss={vl:.4f} (bottom={vl_b:.4f}, top={vl_t:.4f}) "
                  f"train_loss_b={avg_b:.4f}, train_loss_t={avg_t:.4f}")

    # ===== Load best state =====
    if best_st:
        model.load_state_dict(best_st)
        print(f"  Loaded best checkpoint (val_loss={best_vl:.4f})")
    else:
        print(f"  [ERROR] No valid checkpoint saved! Model weights may be random or NaN.")
        print(f"  Attempting to use current model state...")
        # Check if current model state is usable
        if not check_model_health(model, "CNN (last epoch)"):
            print(f"  [FATAL] CNN training completely failed. No valid weights available.")
            return None

    # ===== Verify health after loading =====
    if not check_model_health(model, "CNN (best checkpoint)"):
        print(f"  [FATAL] Best checkpoint has NaN weights!")
        return None

    # ===== Evaluate accuracy =====
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        pb = torch.sigmoid(model(Xdv_t)[0]).cpu().numpy()
        pt = torch.sigmoid(model(Xuv_t)[1]).cpu().numpy()

    # Check for NaN in predictions
    if np.isnan(pb).any() or np.isnan(pt).any():
        print(f"  [ERROR] Model produces NaN predictions even with verified weights!")
        print(f"  pb NaN: {np.isnan(pb).sum()}/{len(pb)}, pt NaN: {np.isnan(pt).sum()}/{len(pt)}")
        return None

    acc_b = np.mean((pb > 0.5).astype(int) == yd_v) * 100
    acc_t = np.mean((pt > 0.5).astype(int) == yu_v) * 100
    print(f"  Bottom acc: {acc_b:.1f}% (N={len(yd_v)})")
    print(f"  Top acc:    {acc_t:.1f}% (N={len(yu_v)})")

    # Additional stats
    print(f"  prob_bottom: mean={pb.mean():.4f}, std={pb.std():.4f}, "
          f"min={pb.min():.4f}, max={pb.max():.4f}")
    print(f"  prob_top:    mean={pt.mean():.4f}, std={pt.std():.4f}, "
          f"min={pt.min():.4f}, max={pt.max():.4f}")

    return model


# ============================================================
# MAIN
# ============================================================
def main():
    print(f"\n{'='*60}")
    print(f"  Loading data for all tickers...")
    print(f"{'='*60}")

    vol_features_all = []
    vol_targets_all = []
    vol_block_rv_all = []
    vol_n_blocks_all = []
    cnn_ticker_data = {}

    for ticker in TICKERS:
        df = load_ticker(ticker)
        if df is None:
            continue

        n = len(df)
        train_end_idx = int(n * TRAIN_RATIO)
        print(f"  {ticker}: {n} bars, train_end={train_end_idx}")

        # Vol: HAR features
        features, target, block_rv, n_blocks = compute_har_features(df)
        vol_features_all.append(features)
        vol_targets_all.append(target)
        vol_block_rv_all.append(block_rv)
        vol_n_blocks_all.append(n_blocks)

        # CNN: 16 features
        features_16 = compute_cnn_features_16(df)
        cnn_ticker_data[ticker] = (df, features_16, train_end_idx)

    # --- Train Vol LSTM ---
    vol_model = train_vol_model(vol_features_all, vol_targets_all,
                                vol_block_rv_all, vol_n_blocks_all)
    vol_path = os.path.join(SAVE_DIR, "vol_lstm_v3.pt")
    torch.save(vol_model.state_dict(), vol_path)
    print(f"\n  ✓ Vol LSTM saved to {vol_path}")

    # --- Verify Vol LSTM saved weights ---
    verify_saved_weights(vol_path, VolLSTM,
                        {'input_size': VOL_N_FEATURES, 'hidden_size': 64,
                         'num_layers': 2, 'dropout': 0.2})

    # --- Train CNN Dual ---
    cnn_model = train_cnn_dual(cnn_ticker_data)
    if cnn_model is not None:
        cnn_path = os.path.join(SAVE_DIR, "cnn_dual.pt")
        torch.save(cnn_model.state_dict(), cnn_path)
        print(f"  ✓ CNN Dual saved to {cnn_path}")

        # --- FIX: Verify CNN saved weights ---
        verified = verify_saved_weights(
            cnn_path, CNNDualModel,
            {'n_features': CNN_N_FEATURES, 'window': CNN_WINDOW})
        if not verified:
            print(f"\n  ✗ CRITICAL: CNN weights failed verification!")
            print(f"    The saved cnn_dual.pt is CORRUPT.")
            print(f"    GMADL Step 4 will NOT work correctly.")
            print(f"    Please investigate and retrain.")
        else:
            print(f"\n  ✓ CNN weights pass all verification checks")
    else:
        print(f"\n  ✗ CNN training failed, no model saved")

    print(f"\n{'='*60}")
    print(f"  DONE! Models saved to {SAVE_DIR}")
    print(f"  Now run GMADL Step 4 — it will load these weights.")
    print(f"{'='*60}")


main()

Device: cuda

  Loading data for all tickers...
  AAPL: 39289 bars, train_end=31431
  MSFT: 39154 bars, train_end=31323
  GOOGL: 35495 bars, train_end=28396
  GOOG: 34511 bars, train_end=27608
  NVDA: 39057 bars, train_end=31245
  TSLA: 39241 bars, train_end=31392
  SPY: 39251 bars, train_end=31400
  QQQ: 39316 bars, train_end=31452

  Training Vol LSTM (pooled across 8 tickers)
  NaN check after scaler: X_tr: 0, y_tr: 0, X_v: 0, y_v: 0
  Train sequences: 10011, Val sequences: 2387
  Epoch 20: val_loss=0.414095, best=0.390796
  Early stop at epoch 23
  Vol LSTM DirAcc: 80.5% (N=2387)
  Best val_loss: 0.390796
  Vol LSTM weight check: ✓ HEALTHY (56641 params)

  ✓ Vol LSTM saved to models/vol_lstm_v3.pt
  ✓ Saved weights verified: models/vol_lstm_v3.pt

  Training CNN Dual Model (FIXED)
  AAPL: 4153 down, 4544 up positions
  MSFT: 3582 down, 3968 up positions
  GOOGL: 3792 down, 3950 up positions
  GOOG: 3721 down, 3906 up positions
  NVDA: 5805 down, 6614 up positions
  TSLA: 7353 down

In [ ]:
"""
诊断Cell — 粘贴在重训cell后面运行
(模型已训练好，只跑诊断)
"""

from scipy import stats

@torch.no_grad()
def run_cnn_inference_v2(df, cnn_model_path, window=30, n_features=16, device='cpu'):
    """Fixed CNN inference: loads scaler from model file."""
    features_16 = compute_cnn_features_16(df)

    checkpoint = torch.load(cnn_model_path, map_location=device, weights_only=True)

    model = CNNDualModel(n_features=n_features, window=window)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    model.to(device)

    scaler_mean = checkpoint['scaler_mean']
    scaler_scale = checkpoint['scaler_scale']

    features_normed = (features_16 - scaler_mean) / (scaler_scale + 1e-8)
    features_normed = np.nan_to_num(features_normed, nan=0.0, posinf=0.0, neginf=0.0)

    prob_bottom = np.full(len(df), np.nan)
    prob_top = np.full(len(df), np.nan)

    batch_size = 512
    for start in range(window, len(features_normed), batch_size):
        end = min(start + batch_size, len(features_normed))
        batch_windows = []
        batch_indices = []
        for i in range(start, end):
            w = features_normed[i - window:i]
            if w.shape == (window, n_features):
                batch_windows.append(w)
                batch_indices.append(i)

        if not batch_windows:
            continue

        x = torch.FloatTensor(np.array(batch_windows)).unsqueeze(1).to(device)
        pb, pt = model(x)

        for j, idx in enumerate(batch_indices):
            prob_bottom[idx] = pb[j].item()
            prob_top[idx] = pt[j].item()

    features = pd.DataFrame(index=df.index)
    pb_logits = np.nan_to_num(prob_bottom, nan=0.0)
    pt_logits = np.nan_to_num(prob_top, nan=0.0)
    features['cnn_prob_bottom'] = 1 / (1 + np.exp(-pb_logits))
    features['cnn_prob_top'] = 1 / (1 + np.exp(-pt_logits))
    features['cnn_reversal_net'] = features['cnn_prob_bottom'] - features['cnn_prob_top']
    features['cnn_max_prob'] = np.maximum(
        features['cnn_prob_bottom'].values, features['cnn_prob_top'].values)

    valid_mask = ~np.isnan(prob_bottom)
    if valid_mask.sum() > 0:
        print(f"  CNN prob_bottom: mean={features['cnn_prob_bottom'][valid_mask].mean():.3f}, "
              f"std={features['cnn_prob_bottom'][valid_mask].std():.3f}, "
              f"range=[{features['cnn_prob_bottom'][valid_mask].min():.3f}, "
              f"{features['cnn_prob_bottom'][valid_mask].max():.3f}]")
        print(f"  CNN prob_top:    mean={features['cnn_prob_top'][valid_mask].mean():.3f}, "
              f"std={features['cnn_prob_top'][valid_mask].std():.3f}, "
              f"range=[{features['cnn_prob_top'][valid_mask].min():.3f}, "
              f"{features['cnn_prob_top'][valid_mask].max():.3f}]")

    return features


def diagnose_cnn_direction(stock, df, cnn_features):
    """Test if CNN signals predict price direction."""
    print(f"\n{'='*60}")
    print(f"  {stock}: CNN Direction Diagnostic")
    print(f"{'='*60}")

    close = df['close'].values
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

    for horizon in [1, 3, 6, 12]:
        future_ret = pd.Series(close).pct_change(horizon).shift(-horizon)

        print(f"\n  --- horizon={horizon} bars ({horizon*15}min) ---")
        print(f"  {'Type':<8} {'Thr':>5} {'N':>6} {'DA':>8} {'p':>10} {'Sig':>5} {'AvgRet':>10}")

        for thr in thresholds:
            # Bottom → price goes up?
            mask = cnn_features['cnn_prob_bottom'] > thr
            n = mask.sum()
            if n >= 15:
                rets = future_ret[mask].dropna()
                if len(rets) >= 10:
                    n_up = int((rets > 0).sum())
                    da = n_up / len(rets)
                    p_val = stats.binomtest(n_up, len(rets), 0.5, alternative='greater').pvalue
                    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    print(f"  {'Bottom':<8} {thr:>5.1f} {len(rets):>6} {da:>8.4f} {p_val:>10.4f} {sig:>5} {rets.mean():>10.6f}")

            # Top → price goes down?
            mask = cnn_features['cnn_prob_top'] > thr
            n = mask.sum()
            if n >= 15:
                rets = future_ret[mask].dropna()
                if len(rets) >= 10:
                    n_down = int((rets < 0).sum())
                    da = n_down / len(rets)
                    p_val = stats.binomtest(n_down, len(rets), 0.5, alternative='greater').pvalue
                    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                    print(f"  {'Top':<8} {thr:>5.1f} {len(rets):>6} {da:>8.4f} {p_val:>10.4f} {sig:>5} {rets.mean():>10.6f}")


# Run diagnostic
cnn_v2_path = os.path.join(SAVE_DIR, "cnn_dual_v2.pt")

for stock in ['AAPL', 'MSFT', 'SPY']:
    print(f"\n\n{'#'*60}")
    print(f"# {stock}")
    print(f"{'#'*60}")

    df = load_ticker(stock)
    if df is None:
        continue

    n = len(df)
    test_start = int(n * TRAIN_RATIO)
    df_test = df.iloc[test_start:].reset_index(drop=True)
    print(f"  Test: {len(df_test)} bars (from bar {test_start})")

    print(f"  Running CNN inference (v2 with saved scaler)...")
    cnn_features = run_cnn_inference_v2(
        df_test, cnn_v2_path, CNN_WINDOW, CNN_N_FEATURES, str(DEVICE))

    diagnose_cnn_direction(stock, df_test, cnn_features)

print(f"\n\n{'='*60}")
print(f"结果解读:")
print(f"{'='*60}")
print(f"  DA > 55% + p < 0.05 → CNN有方向信号! 可以整合进LSTM")
print(f"  DA ≈ 50% everywhere → CNN转折点不能预测方向")
print(f"  horizon=6最好 → 改GMADL的RETURN_HORIZON=6匹配CNN")



############################################################
# AAPL
############################################################
  Test: 7858 bars (from bar 31431)
  Running CNN inference (v2 with saved scaler)...
  CNN prob_bottom: mean=0.232, std=0.127, range=[0.025, 0.795]
  CNN prob_top:    mean=0.166, std=0.101, range=[0.010, 0.766]

  AAPL: CNN Direction Diagnostic

  --- horizon=1 bars (15min) ---
  Type       Thr      N       DA          p   Sig     AvgRet
  Bottom     0.5    360   0.4750     0.8417    ns  -0.000087
  Top        0.5     83   0.4458     0.8639    ns   0.000461
  Bottom     0.6    157   0.4777     0.7383    ns   0.000007
  Top        0.6     45   0.5111     0.5000    ns  -0.000109
  Bottom     0.7     52   0.5000     0.5551    ns   0.000314
  Top        0.7     16   0.6250     0.2272    ns  -0.001319

  --- horizon=3 bars (45min) ---
  Type       Thr      N       DA          p   Sig     AvgRet
  Bottom     0.5    360   0.4972     0.5628    ns  -0.000127
  Top  

In [ ]:
"""
分层信号策略: 正确使用 Vol + CNN + 技术指标
=============================================

核心思路:
  CNN不是"特征"，是"条件交易信号"
  Vol不是"方向预测"，是"仓位大小"

决策流程:
  1. 趋势检测: 过去6bar涨/跌>0.5%?
  2. 如果在趋势中 → CNN预测是否反转
  3. 如果CNN说反转 → 交易 (方向=反转方向)
  4. Vol预测高波动 → 加大仓位 (反转幅度更大)
  5. 如果不在趋势中 → 不交易 (CNN无效)

这完全不需要LSTM来整合信号。
"""

import os
import numpy as np
import pandas as pd
import torch
from scipy import stats


# ============================================================
# PART 1: 分层信号策略
# ============================================================

class LayeredSignalStrategy:
    """
    分层决策:
      Layer 1: 趋势检测 (规则)
      Layer 2: CNN反转预测 (模型)
      Layer 3: Vol仓位调整 (模型)
    """

    def __init__(self,
                 trend_lookback=6,
                 trend_pct=0.5,
                 cnn_threshold=0.5,
                 hold_bars=6,
                 vol_percentile=75):
        self.trend_lookback = trend_lookback
        self.trend_pct = trend_pct / 100.0
        self.cnn_threshold = cnn_threshold
        self.hold_bars = hold_bars  # 持仓时间 = CNN lookahead
        self.vol_percentile = vol_percentile

    def detect_trends(self, close):
        """Layer 1: 趋势检测 (和CNN训练条件一致)"""
        n = len(close)
        trend = np.zeros(n)  # 0=无趋势, -1=下跌, +1=上涨
        for t in range(self.trend_lookback, n):
            move = (close[t] - close[t - self.trend_lookback]) / (close[t - self.trend_lookback] + 1e-10)
            if move < -self.trend_pct:
                trend[t] = -1  # 下跌趋势
            elif move > self.trend_pct:
                trend[t] = 1   # 上涨趋势
        return trend

    def generate_signals(self, df, cnn_features, vol_features=None):
        """
        生成交易信号。

        返回:
          signals: +1=做多, -1=做空, 0=不交易
          confidence: CNN概率值 (用于仓位调整)
          vol_scale: Vol预测值 (用于仓位调整)
        """
        close = df['close'].values
        n = len(close)

        trend = self.detect_trends(close)
        signals = np.zeros(n)
        confidence = np.zeros(n)
        vol_scale = np.ones(n)

        # Vol gate
        if vol_features is not None:
            vol_pred = vol_features['vol_pred_rv'].values
            vol_thr = np.percentile(vol_pred[vol_pred > 0], self.vol_percentile)
            vol_scale = vol_pred / (np.median(vol_pred[vol_pred > 0]) + 1e-10)

        prob_bottom = cnn_features['cnn_prob_bottom'].values
        prob_top = cnn_features['cnn_prob_top'].values

        # 信号生成 (带冷却期防止重复信号)
        cooldown = 0
        for t in range(self.trend_lookback, n):
            if cooldown > 0:
                cooldown -= 1
                continue

            if trend[t] == -1 and prob_bottom[t] > self.cnn_threshold:
                # 下跌趋势 + CNN说底部反转 → 做多
                signals[t] = 1
                confidence[t] = prob_bottom[t]
                cooldown = self.hold_bars  # 持仓期间不开新仓

            elif trend[t] == 1 and prob_top[t] > self.cnn_threshold:
                # 上涨趋势 + CNN说顶部反转 → 做空
                signals[t] = -1
                confidence[t] = prob_top[t]
                cooldown = self.hold_bars

        return signals, confidence, vol_scale

    def backtest(self, df, signals, confidence, vol_scale):
        """
        简单回测: 每个信号持仓hold_bars根bar。

        返回详细的交易记录和统计。
        """
        close = df['close'].values
        n = len(close)
        trades = []

        for t in range(n):
            if signals[t] == 0:
                continue
            if t + self.hold_bars >= n:
                continue

            entry_price = close[t]
            exit_price = close[t + self.hold_bars]
            raw_ret = (exit_price - entry_price) / entry_price

            # 做空时收益取反
            if signals[t] == -1:
                raw_ret = -raw_ret

            trades.append({
                'bar': t,
                'direction': 'LONG' if signals[t] == 1 else 'SHORT',
                'entry': entry_price,
                'exit': exit_price,
                'return': raw_ret,
                'confidence': confidence[t],
                'vol_scale': vol_scale[t],
            })

        return pd.DataFrame(trades)


# ============================================================
# PART 2: 评估和诊断
# ============================================================

def evaluate_strategy(stock, df, cnn_features, vol_features=None):
    """完整评估分层策略"""

    print(f"\n{'#'*70}")
    print(f"# {stock}: Layered Signal Strategy")
    print(f"{'#'*70}")

    close = df['close'].values
    n = len(close)

    # 趋势统计
    strategy = LayeredSignalStrategy()
    trend = strategy.detect_trends(close)
    n_down = (trend == -1).sum()
    n_up = (trend == 1).sum()
    n_none = (trend == 0).sum()
    print(f"\n  趋势分布: 下跌={n_down} ({n_down/n*100:.1f}%), "
          f"上涨={n_up} ({n_up/n*100:.1f}%), "
          f"无趋势={n_none} ({n_none/n*100:.1f}%)")

    # 不同阈值测试
    print(f"\n  {'Thr':>5} {'Hold':>5} {'N trades':>8} {'Win%':>8} {'Avg Ret':>10} "
          f"{'Total Ret':>10} {'p-value':>10} {'Sig':>5}")
    print(f"  {'-'*5} {'-'*5} {'-'*8} {'-'*8} {'-'*10} {'-'*10} {'-'*10} {'-'*5}")

    for cnn_thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
        for hold in [3, 6, 12]:
            strat = LayeredSignalStrategy(
                cnn_threshold=cnn_thr,
                hold_bars=hold
            )
            signals, conf, vs = strat.generate_signals(df, cnn_features, vol_features)
            trades = strat.backtest(df, signals, conf, vs)

            if len(trades) >= 10:
                win_rate = (trades['return'] > 0).mean()
                avg_ret = trades['return'].mean()
                total_ret = trades['return'].sum()
                n_win = int((trades['return'] > 0).sum())
                p_val = stats.binomtest(n_win, len(trades), 0.5, alternative='greater').pvalue
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                print(f"  {cnn_thr:>5.1f} {hold:>5} {len(trades):>8} {win_rate:>8.4f} "
                      f"{avg_ret:>10.6f} {total_ret:>10.4f} {p_val:>10.4f} {sig:>5}")

    # 详细分析: 最佳参数
    print(f"\n  === 详细分析 (thr=0.5, hold=6) ===")
    strat = LayeredSignalStrategy(cnn_threshold=0.5, hold_bars=6)
    signals, conf, vs = strat.generate_signals(df, cnn_features, vol_features)
    trades = strat.backtest(df, signals, conf, vs)

    if len(trades) >= 5:
        long_trades = trades[trades['direction'] == 'LONG']
        short_trades = trades[trades['direction'] == 'SHORT']

        if len(long_trades) >= 5:
            lw = (long_trades['return'] > 0).mean()
            lr = long_trades['return'].mean()
            print(f"  LONG trades:  N={len(long_trades)}, win={lw:.4f}, avg_ret={lr:.6f}")

        if len(short_trades) >= 5:
            sw = (short_trades['return'] > 0).mean()
            sr = short_trades['return'].mean()
            print(f"  SHORT trades: N={len(short_trades)}, win={sw:.4f}, avg_ret={sr:.6f}")

        # 高置信度 vs 低置信度
        if len(trades) >= 20:
            median_conf = trades['confidence'].median()
            high_conf = trades[trades['confidence'] > median_conf]
            low_conf = trades[trades['confidence'] <= median_conf]
            print(f"\n  High confidence (>{median_conf:.2f}): N={len(high_conf)}, "
                  f"win={( high_conf['return'] > 0).mean():.4f}, "
                  f"avg_ret={high_conf['return'].mean():.6f}")
            print(f"  Low confidence  (<={median_conf:.2f}): N={len(low_conf)}, "
                  f"win={(low_conf['return'] > 0).mean():.4f}, "
                  f"avg_ret={low_conf['return'].mean():.6f}")

    # 和基线对比: 如果每次下跌趋势都做多(不用CNN)
    print(f"\n  === 基线对比 ===")
    for hold in [3, 6, 12]:
        # 基线: 每次下跌趋势bar都做多
        baseline_mask = trend == -1
        future_ret = pd.Series(close).pct_change(hold).shift(-hold)
        base_rets = future_ret[baseline_mask].dropna()
        if len(base_rets) >= 10:
            base_win = (base_rets > 0).mean()
            base_avg = base_rets.mean()
            print(f"  基线(下跌→全部做多, hold={hold}): N={len(base_rets)}, "
                  f"win={base_win:.4f}, avg_ret={base_avg:.6f}")

        # CNN过滤后
        cnn_mask = (trend == -1) & (cnn_features['cnn_prob_bottom'] > 0.5)
        cnn_rets = future_ret[cnn_mask].dropna()
        if len(cnn_rets) >= 10:
            cnn_win = (cnn_rets > 0).mean()
            cnn_avg = cnn_rets.mean()
            lift = cnn_win / max(base_win, 1e-6)
            print(f"  CNN过滤(prob>0.5, hold={hold}):   N={len(cnn_rets)}, "
                  f"win={cnn_win:.4f}, avg_ret={cnn_avg:.6f}, lift={lift:.2f}x")


# ============================================================
# PART 3: 精确复现训练评估 (precision/recall)
# ============================================================
def diagnose_precision(stock, df, cnn_features):
    """用和训练完全一致的方式评估CNN在测试集上的表现"""
    close = df['close'].values
    n = len(close)
    LA = CNN_LOOKAHEAD
    rev_pct = TREND_PCT / 100.0

    print(f"\n{'='*70}")
    print(f"  {stock}: 训练标签精确度复现")
    print(f"{'='*70}")

    # 复现训练标签
    positions_down = []
    positions_up = []

    for t in range(max(CNN_WINDOW, TREND_LOOKBACK), n - LA):
        p_now = close[t]
        p_past = close[t - TREND_LOOKBACK]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        if move < -(TREND_PCT / 100.0):
            future_max = np.max(close[t+1:t+1+LA])
            label = 1 if (future_max - p_now) / p_now > rev_pct else 0
            prob = cnn_features['cnn_prob_bottom'].iloc[t] if t < len(cnn_features) else 0.5
            positions_down.append({'idx': t, 'label': label, 'prob': prob})

        elif move > (TREND_PCT / 100.0):
            future_min = np.min(close[t+1:t+1+LA])
            label = 1 if (p_now - future_min) / p_now > rev_pct else 0
            prob = cnn_features['cnn_prob_top'].iloc[t] if t < len(cnn_features) else 0.5
            positions_up.append({'idx': t, 'label': label, 'prob': prob})

    # Bottom
    if positions_down:
        pdf_d = pd.DataFrame(positions_down)
        base_rate = pdf_d['label'].mean()
        overall_acc = ((pdf_d['prob'] > 0.5).astype(int) == pdf_d['label']).mean()
        print(f"\n  Bottom (下跌趋势bar):")
        print(f"    样本数: {len(pdf_d)}, 正样本率: {base_rate:.3f}")
        print(f"    整体准确率 (prob>0.5): {overall_acc:.3f}")
        print(f"    训练时验证准确率:      0.737")
        print(f"")
        print(f"    {'Thr':>5} {'N pred+':>8} {'Precision':>10} {'Base rate':>10} {'Lift':>6} {'Recall':>8}")

        for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
            pred_pos = pdf_d[pdf_d['prob'] > thr]
            if len(pred_pos) >= 5:
                prec = pred_pos['label'].mean()
                recall = pred_pos['label'].sum() / max(pdf_d['label'].sum(), 1)
                lift = prec / max(base_rate, 1e-6)
                print(f"    {thr:>5.1f} {len(pred_pos):>8} {prec:>10.4f} {base_rate:>10.4f} "
                      f"{lift:>6.2f}x {recall:>8.4f}")

    # Top
    if positions_up:
        pdf_u = pd.DataFrame(positions_up)
        base_rate = pdf_u['label'].mean()
        overall_acc = ((pdf_u['prob'] > 0.5).astype(int) == pdf_u['label']).mean()
        print(f"\n  Top (上涨趋势bar):")
        print(f"    样本数: {len(pdf_u)}, 正样本率: {base_rate:.3f}")
        print(f"    整体准确率 (prob>0.5): {overall_acc:.3f}")
        print(f"    训练时验证准确率:      0.797")
        print(f"")
        print(f"    {'Thr':>5} {'N pred+':>8} {'Precision':>10} {'Base rate':>10} {'Lift':>6} {'Recall':>8}")

        for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
            pred_pos = pdf_u[pdf_u['prob'] > thr]
            if len(pred_pos) >= 5:
                prec = pred_pos['label'].mean()
                recall = pred_pos['label'].sum() / max(pdf_u['label'].sum(), 1)
                lift = prec / max(base_rate, 1e-6)
                print(f"    {thr:>5.1f} {len(pred_pos):>8} {prec:>10.4f} {base_rate:>10.4f} "
                      f"{lift:>6.2f}x {recall:>8.4f}")


# ============================================================
# RUN ALL
# ============================================================
cnn_v2_path = os.path.join(SAVE_DIR, "cnn_dual_v2.pt")

for stock in ['AAPL', 'MSFT', 'SPY']:
    df = load_ticker(stock)
    if df is None:
        continue

    n = len(df)
    test_start = int(n * TRAIN_RATIO)
    df_test = df.iloc[test_start:].reset_index(drop=True)

    cnn_features = run_cnn_inference_v2(
        df_test, cnn_v2_path, CNN_WINDOW, CNN_N_FEATURES, str(DEVICE))

    # 1. 精确度复现 (CNN在训练标签意义下是否仍然准确)
    diagnose_precision(stock, df_test, cnn_features)

    # 2. 分层策略评估 (能不能转化为交易收益)
    evaluate_strategy(stock, df_test, cnn_features)

print(f"\n\n{'='*70}")
print(f"如何解读:")
print(f"{'='*70}")
print(f"  1. precision复现: 如果测试集准确率≈训练验证准确率 → CNN在测试集上work")
print(f"     如果大幅下降 → CNN过拟合了训练数据")
print(f"")
print(f"  2. 分层策略: 如果CNN过滤后win% > 基线win% → CNN确实在筛选更好的交易机会")
print(f"     如果lift > 1.2x → 值得在量化框架中使用")
print(f"     如果win% ≈ 基线 → CNN虽然形态识别准，但不影响交易结果")

  CNN prob_bottom: mean=0.232, std=0.127, range=[0.025, 0.795]
  CNN prob_top:    mean=0.166, std=0.101, range=[0.010, 0.766]

  AAPL: 训练标签精确度复现

  Bottom (下跌趋势bar):
    样本数: 1253, 正样本率: 0.274
    整体准确率 (prob>0.5): 0.705
    训练时验证准确率:      0.737

      Thr  N pred+  Precision  Base rate   Lift   Recall
      0.3      554     0.3809     0.2737   1.39x   0.6152
      0.4      299     0.4181     0.2737   1.53x   0.3644
      0.5      139     0.4029     0.2737   1.47x   0.1633
      0.6       69     0.3768     0.2737   1.38x   0.0758
      0.7       26     0.6154     0.2737   2.25x   0.0466

  Top (上涨趋势bar):
    样本数: 1182, 正样本率: 0.284
    整体准确率 (prob>0.5): 0.717
    训练时验证准确率:      0.797

      Thr  N pred+  Precision  Base rate   Lift   Recall
      0.3      193     0.4715     0.2843   1.66x   0.2708
      0.4       64     0.4375     0.2843   1.54x   0.0833
      0.5       22     0.5455     0.2843   1.92x   0.0357
      0.6       12     0.5000     0.2843   1.76x   0.0179

#################

In [ ]:
"""
GMADL + Multi-Task LSTM for Stock Return Prediction
=====================================================
Based on: Michańków et al. (2024) "Generalized Mean Absolute Directional Loss"
         arXiv:2412.18405

Four-step experiment:
  Step 1: LSTM predicting returns with MSE (baseline)
  Step 2: LSTM predicting returns with GMADL
  Step 3: Multi-task LSTM (GMADL regression + BCE classification)
  Step 4: Multi-task LSTM + Vol/CNN auxiliary features

Usage:
  Upload to Colab, mount Google Drive, adjust DATA_PATH.
  Each step builds on previous, results are compared at the end.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================
class Config:
    # Data — Google Drive splits (1min → resample to 15min)
    DRIVE_BASE = CONFIG['data_dir'] / r'splits'
    FREQ_RAW = "1min"           # raw data frequency
    RESAMPLE_PERIOD = 15        # resample to 15min
    STOCKS = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'SPY', 'QQQ']
    TIMEFRAME = '15min'

    # Returns target
    RETURN_HORIZON = 1  # predict 1-bar-ahead return

    # Sequence
    SEQ_LEN = 60        # 60 bars lookback (15 hours at 15min)

    # Model
    HIDDEN_SIZE = 128
    NUM_LAYERS = 2
    DROPOUT = 0.3

    # Training
    BATCH_SIZE = 256
    EPOCHS = 50
    LR = 1e-3
    PATIENCE = 8

    # GMADL parameters (from paper)
    GMADL_A = 100.0     # sigmoid steepness (reduced from 1000 to avoid saturation on small returns)
    GMADL_B = 1.0        # return magnitude exponent

    # Multi-task
    LAMBDA_CLS = 0.3     # weight for classification loss

    # Data split
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    # TEST_RATIO = 0.15 (remainder)

    # Vol prediction (HAR features)
    VOL_BLOCK_SIZE = 24  # 24 bars of 15min = 6 hours
    VOL_MODEL_PATH = CONFIG['data_dir'] / r'vol_lstm_v3.pt'  # adjust
    VOL_SEQ_LEN = 20     # match V3

    # CNN turning point
    CNN_WINDOW = 30
    CNN_LOOKAHEAD = 6
    CNN_N_FEATURES = 16
    CNN_MODEL_PATH = CONFIG['data_dir'] / r'cnn_dual.pt'  # adjust

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42

cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

# ============================================================
# LOSS FUNCTIONS
# ============================================================

class GMADLLoss(nn.Module):
    """
    Generalized Mean Absolute Directional Loss (GMADL)
    From: Michańków et al. (2024), arXiv:2412.18405
    """
    def __init__(self, a=1000.0, b=1.0, eps=1e-8):
        super().__init__()
        self.a = a
        self.b = b
        self.eps = eps

    def forward(self, pred_return, actual_return):
        product = actual_return * pred_return
        direction_score = torch.sigmoid(self.a * product) - 0.5
        magnitude = (torch.abs(actual_return) + self.eps) ** self.b
        loss_per_sample = (-1.0) * direction_score * magnitude
        return loss_per_sample.mean()


class MultiTaskLoss(nn.Module):
    """Combined loss: GMADL (regression) + BCE (direction classification)"""
    def __init__(self, a=1000.0, b=1.0, lambda_cls=0.3):
        super().__init__()
        self.gmadl = GMADLLoss(a=a, b=b)
        self.bce = nn.BCEWithLogitsLoss()
        self.lambda_cls = lambda_cls

    def forward(self, pred_return, pred_direction_logit, actual_return):
        loss_reg = self.gmadl(pred_return, actual_return)
        direction_target = (actual_return > 0).float()
        loss_cls = self.bce(pred_direction_logit, direction_target)
        return loss_reg + self.lambda_cls * loss_cls, loss_reg, loss_cls


# ============================================================
# DATASET
# ============================================================

def compute_technical_features(df):
    """Compute core technical indicators from OHLCV data."""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    volume = df['volume'].values

    features = pd.DataFrame(index=df.index)

    for h in [1, 5, 15, 30]:
        features[f'ret_{h}'] = pd.Series(close).pct_change(h).values

    ret1 = pd.Series(close).pct_change()
    for w in [10, 30, 60]:
        features[f'vol_{w}'] = ret1.rolling(w).std().values

    delta = ret1.copy()
    gain = delta.clip(lower=0)
    loss_val = (-delta).clip(lower=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss_val.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-10)
    features['rsi_14'] = (100 - 100 / (1 + rs)).values

    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    features['macd'] = macd.values
    features['macd_signal'] = signal.values
    features['macd_hist'] = (macd - signal).values

    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    features['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values

    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)),
        np.abs(low - np.roll(close, 1))
    ))
    features['atr_14'] = pd.Series(tr).rolling(14).mean().values

    features['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)
    features['vol_change'] = pd.Series(volume).pct_change().values

    hh = pd.Series(high).rolling(20).max()
    ll = pd.Series(low).rolling(20).min()
    features['price_pos'] = ((close - ll) / (hh - ll + 1e-10)).values

    return features


# ============================================================
# PRE-TRAINED MODEL ARCHITECTURES (for inference only)
# ============================================================

class VolLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class CNNDualModel(nn.Module):
    """
    CNN for bottom/top reversal detection.
    FIX: Returns raw logits (no sigmoid). Sigmoid applied once in run_cnn_inference().
    """
    def __init__(self, n_features=16, window=30):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, n_features), padding=(1, 0))
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 1), padding=(1, 0))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_bottom = nn.Linear(64, 1)
        self.fc_top = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        # FIX: return logits (matching training code), no double sigmoid
        return self.fc_bottom(x).squeeze(-1), self.fc_top(x).squeeze(-1)


# ============================================================
# VOL/CNN INFERENCE → FEATURES FOR STEP 4
# ============================================================

def compute_har_inputs(df, block_size=24):
    """Compute the 17 features that feed INTO the Vol model (matching V3)."""
    close = df['close'].values
    ret = np.log(close[1:] / close[:-1])
    ret = np.concatenate([[0], ret])

    n = len(ret)
    n_blocks = n // block_size
    usable = n_blocks * block_size
    blocks_ret = ret[:usable].reshape(n_blocks, block_size)

    block_rv_arr = np.array([blocks_ret[i].std() for i in range(n_blocks)])
    block_rv_ssq_arr = np.array([np.sqrt(np.sum(blocks_ret[i]**2)) for i in range(n_blocks)])
    block_agg_ret_arr = np.array([blocks_ret[i].sum() for i in range(n_blocks)])
    block_mean_ret_arr = np.array([blocks_ret[i].mean() for i in range(n_blocks)])

    block_rv = np.full(n, np.nan)
    block_rv_ssq = np.full(n, np.nan)
    block_agg_ret = np.full(n, np.nan)
    block_mean_ret = np.full(n, np.nan)
    for i in range(n_blocks):
        s, e = i * block_size, (i + 1) * block_size
        block_rv[s:e] = block_rv_arr[i]
        block_rv_ssq[s:e] = block_rv_ssq_arr[i]
        block_agg_ret[s:e] = block_agg_ret_arr[i]
        block_mean_ret[s:e] = block_mean_ret_arr[i]

    rv = pd.Series(block_rv)
    har = pd.DataFrame(index=df.index)
    har['block_rv'] = rv
    har['block_rv_ssq'] = block_rv_ssq
    har['block_agg_ret'] = block_agg_ret
    har['block_mean_ret'] = block_mean_ret
    har['rv_d'] = rv
    har['rv_w'] = rv.rolling(5 * block_size, min_periods=block_size).mean()
    har['rv_m'] = rv.rolling(20 * block_size, min_periods=block_size).mean()
    for lag in [1, 2, 3, 5, 10]:
        har[f'rv_lag{lag}'] = rv.shift(lag * block_size)
    har['rv_change'] = rv.pct_change(block_size)
    har['rv_ratio'] = rv / (har['rv_m'] + 1e-10)
    har['vol_of_vol'] = rv.rolling(5 * block_size, min_periods=block_size).std()
    har['neg_ret_frac'] = (pd.Series(block_agg_ret) < 0).astype(float)
    har['abs_block_ret'] = pd.Series(block_agg_ret).abs()
    return har


def compute_cnn_inputs(df, window=30, n_features=16):
    """Compute the OHLCV-based feature windows that feed INTO the CNN."""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    volume = df['volume'].values

    feat = pd.DataFrame(index=df.index)
    ret = pd.Series(close).pct_change()

    feat['ret_1'] = ret.values
    feat['ret_5'] = ret.rolling(5).sum().values
    feat['ret_15'] = ret.rolling(15).sum().values
    feat['high_low_range'] = (high - low) / (close + 1e-10)
    feat['close_position'] = (close - low) / (high - low + 1e-10)

    sma5 = pd.Series(close).rolling(5).mean()
    sma15 = pd.Series(close).rolling(15).mean()
    sma30 = pd.Series(close).rolling(30).mean()
    feat['ma5_15'] = ((sma5 - sma15) / (sma15 + 1e-10)).values
    feat['ma5_30'] = ((sma5 - sma30) / (sma30 + 1e-10)).values

    feat['vol_5'] = ret.rolling(5).std().values
    feat['vol_15'] = ret.rolling(15).std().values

    delta = ret.copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss_v = (-delta).clip(lower=0).rolling(14).mean()
    feat['rsi'] = (100 - 100 / (1 + gain / (loss_v + 1e-10))).values

    feat['vol_ratio'] = volume / (pd.Series(volume).rolling(20).mean().values + 1e-10)

    sma20 = pd.Series(close).rolling(20).mean()
    std20 = pd.Series(close).rolling(20).std()
    feat['bb_pos'] = ((close - sma20) / (2 * std20 + 1e-10)).values

    ema12 = pd.Series(close).ewm(span=12).mean()
    ema26 = pd.Series(close).ewm(span=26).mean()
    feat['macd'] = ((ema12 - ema26) / (close + 1e-10)).values

    tr = np.maximum(high - low, np.maximum(
        np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    feat['atr'] = (pd.Series(tr).rolling(14).mean() / (close + 1e-10)).values

    cols = list(feat.columns)[:n_features]
    while len(cols) < n_features:
        cols.append(cols[-1])
    feat = feat[cols]
    feat.columns = [f'cnn_in_{i}' for i in range(n_features)]
    return feat


@torch.no_grad()
def run_vol_inference(df, vol_model_path, block_size=24, seq_len=10, device='cpu'):
    """Load trained Vol LSTM and run inference on full dataset."""
    har = compute_har_inputs(df, block_size)
    har_valid = har.dropna()

    if not os.path.exists(vol_model_path):
        print(f"  [WARNING] Vol model not found at {vol_model_path}")
        print(f"  → Using HAR raw features as fallback (not model predictions)")
        return _vol_fallback_features(df, block_size)

    model = VolLSTM(input_size=17, hidden_size=64, num_layers=2, dropout=0.2)
    state = torch.load(vol_model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    model.to(device)

    har_vals = har_valid.values.astype(np.float32)
    har_mean = har_vals.mean(axis=0)
    har_std = har_vals.std(axis=0) + 1e-8
    har_normed = (har_vals - har_mean) / har_std

    predictions = np.full(len(df), np.nan)
    valid_indices = har_valid.index.values
    for i in range(seq_len, len(har_normed)):
        seq = har_normed[i - seq_len:i]
        x = torch.FloatTensor(seq).unsqueeze(0).to(device)
        pred = model(x).item()
        orig_idx = valid_indices[i]
        predictions[orig_idx] = pred

    pred_series = pd.Series(predictions, index=df.index)
    features = pd.DataFrame(index=df.index)
    features['vol_pred_rv'] = pred_series
    features['vol_current_rv'] = har['rv_d']
    features['vol_ratio_pred'] = pred_series / (har['rv_d'] + 1e-10)
    features['vol_pred_direction'] = (pred_series > har['rv_d']).astype(float)
    return features


@torch.no_grad()
def run_cnn_inference(df, cnn_model_path, window=30, n_features=16, device='cpu'):
    """
    Load trained CNN dual model and run inference on full dataset.
    FIX: model.forward() now returns logits, sigmoid applied here ONCE.
    """
    cnn_inputs = compute_cnn_inputs(df, window, n_features)
    cnn_valid = cnn_inputs.dropna()

    if not os.path.exists(cnn_model_path):
        print(f"  [WARNING] CNN model not found at {cnn_model_path}")
        print(f"  → Using trend features as fallback (not model predictions)")
        return _cnn_fallback_features(df, window)

    model = CNNDualModel(n_features=n_features, window=window)
    state = torch.load(cnn_model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    model.to(device)

    cnn_vals = cnn_valid.values.astype(np.float32)
    cnn_mean = cnn_vals.mean(axis=0)
    cnn_std = cnn_vals.std(axis=0) + 1e-8
    cnn_normed = (cnn_vals - cnn_mean) / cnn_std

    prob_bottom = np.full(len(df), np.nan)
    prob_top = np.full(len(df), np.nan)
    valid_indices = cnn_valid.index.values

    batch_size = 512
    for start in range(window, len(cnn_normed), batch_size):
        end = min(start + batch_size, len(cnn_normed))
        batch_windows = []
        batch_indices = []
        for i in range(start, end):
            w = cnn_normed[i - window:i]
            batch_windows.append(w)
            batch_indices.append(valid_indices[i])

        x = torch.FloatTensor(np.array(batch_windows)).unsqueeze(1).to(device)
        pb, pt = model(x)  # FIX: now returns logits

        for j, idx in enumerate(batch_indices):
            prob_bottom[idx] = pb[j].item()
            prob_top[idx] = pt[j].item()

    # FIX: Apply sigmoid ONCE (logits → probabilities)
    # NaN → logit=0 → sigmoid(0)=0.5 (neutral)
    features = pd.DataFrame(index=df.index)
    pb_logits = np.nan_to_num(prob_bottom, nan=0.0)
    pt_logits = np.nan_to_num(prob_top, nan=0.0)
    features['cnn_prob_bottom'] = 1 / (1 + np.exp(-pb_logits))
    features['cnn_prob_top'] = 1 / (1 + np.exp(-pt_logits))
    features['cnn_reversal_net'] = features['cnn_prob_bottom'] - features['cnn_prob_top']
    features['cnn_max_prob'] = np.maximum(
        features['cnn_prob_bottom'].values, features['cnn_prob_top'].values)

    # Diagnostic: check CNN output distribution
    valid_mask = ~np.isnan(prob_bottom)
    if valid_mask.sum() > 0:
        print(f"  CNN prob_bottom: mean={features['cnn_prob_bottom'][valid_mask].mean():.3f}, "
              f"std={features['cnn_prob_bottom'][valid_mask].std():.3f}, "
              f"range=[{features['cnn_prob_bottom'][valid_mask].min():.3f}, "
              f"{features['cnn_prob_bottom'][valid_mask].max():.3f}]")
        print(f"  CNN prob_top:    mean={features['cnn_prob_top'][valid_mask].mean():.3f}, "
              f"std={features['cnn_prob_top'][valid_mask].std():.3f}, "
              f"range=[{features['cnn_prob_top'][valid_mask].min():.3f}, "
              f"{features['cnn_prob_top'][valid_mask].max():.3f}]")

    return features


def _vol_fallback_features(df, block_size=24):
    har = compute_har_inputs(df, block_size)
    features = pd.DataFrame(index=df.index)
    features['vol_pred_rv'] = har['rv_d']
    features['vol_current_rv'] = har['rv_d']
    features['vol_ratio_pred'] = har['rv_ratio']
    features['vol_pred_direction'] = (har['rv_change'] > 0).astype(float)
    return features


def _cnn_fallback_features(df, window=30):
    close = df['close'].values
    ret = pd.Series(close).pct_change()
    features = pd.DataFrame(index=df.index)
    mom_15 = ret.rolling(15).sum()
    mom_std = mom_15.rolling(60).std()
    z_score = -mom_15 / (mom_std + 1e-10)
    features['cnn_prob_bottom'] = (1 / (1 + np.exp(-z_score))).values
    features['cnn_prob_top'] = (1 / (1 + np.exp(z_score))).values
    features['cnn_reversal_net'] = (features['cnn_prob_bottom'] - features['cnn_prob_top']).values
    features['cnn_max_prob'] = np.maximum(
        features['cnn_prob_bottom'].values, features['cnn_prob_top'].values)
    return features


class ReturnPredictionDataset(Dataset):
    def __init__(self, features_array, returns_array, seq_len):
        self.features = features_array
        self.returns = returns_array
        self.seq_len = seq_len
        self.valid_len = len(returns_array) - seq_len

    def __len__(self):
        return self.valid_len

    def __getitem__(self, idx):
        x = self.features[idx:idx + self.seq_len]
        y = self.returns[idx + self.seq_len]
        return torch.FloatTensor(x), torch.FloatTensor([y])


# ============================================================
# MODELS
# ============================================================

class LSTMReturnPredictor(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2,
                 dropout=0.3, mode='regression_only'):
        super().__init__()
        self.mode = mode
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
        )
        if mode == 'multi_task':
            self.cls_head = nn.Sequential(
                nn.Linear(hidden_size, 64), nn.ReLU(),
                nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
            )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        h = lstm_out[:, -1, :]
        h = self.layer_norm(h)
        h = self.dropout(h)
        pred_return = self.reg_head(h)
        if self.mode == 'multi_task':
            pred_dir = self.cls_head(h)
            return pred_return, pred_dir
        return pred_return


class AttentionLSTMPredictor(nn.Module):
    """Step 5: LSTM with Feature-level Attention."""
    def __init__(self, n_features, hidden_size=128, num_layers=2,
                 dropout=0.3, mode='multi_task'):
        super().__init__()
        self.mode = mode
        self.hidden_size = hidden_size
        self.feature_gate = nn.Sequential(
            nn.Linear(n_features, n_features), nn.Sigmoid()
        )
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attn_w = nn.Linear(hidden_size, 1, bias=False)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
        )
        if mode == 'multi_task':
            self.cls_head = nn.Sequential(
                nn.Linear(hidden_size, 64), nn.ReLU(),
                nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
            )

    def forward(self, x):
        gate = self.feature_gate(x)
        x_gated = x * gate
        lstm_out, _ = self.lstm(x_gated)
        attn_scores = self.attn_w(lstm_out).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1)
        h = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)
        h = self.layer_norm(h)
        h = self.dropout(h)
        pred_return = self.reg_head(h)
        if self.mode == 'multi_task':
            pred_dir = self.cls_head(h)
            return pred_return, pred_dir
        return pred_return


class DualStreamLSTMPredictor(nn.Module):
    """Step 6: Dual-Stream LSTM."""
    def __init__(self, n_base_features, n_aux_features, hidden_size=128,
                 num_layers=2, dropout=0.3, mode='multi_task'):
        super().__init__()
        self.mode = mode
        self.n_base = n_base_features
        self.n_aux = n_aux_features
        self.lstm_base = nn.LSTM(
            input_size=n_base_features, hidden_size=hidden_size // 2,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.lstm_aux = nn.LSTM(
            input_size=n_aux_features, hidden_size=hidden_size // 2,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
        )
        if mode == 'multi_task':
            self.cls_head = nn.Sequential(
                nn.Linear(hidden_size, 64), nn.ReLU(),
                nn.Dropout(dropout * 0.5), nn.Linear(64, 1)
            )

    def forward(self, x):
        x_base = x[:, :, :self.n_base]
        x_aux = x[:, :, self.n_base:]
        out_base, _ = self.lstm_base(x_base)
        out_aux, _ = self.lstm_aux(x_aux)
        h_base = out_base[:, -1, :]
        h_aux = out_aux[:, -1, :]
        h = torch.cat([h_base, h_aux], dim=1)
        h = self.layer_norm(h)
        h = self.dropout(h)
        pred_return = self.reg_head(h)
        if self.mode == 'multi_task':
            pred_dir = self.cls_head(h)
            return pred_return, pred_dir
        return pred_return


# ============================================================
# DATA LOADING & PREPARATION
# ============================================================

def load_split_csv(csv_path):
    """Load a single split CSV (train.csv / test.csv)."""
    df = pd.read_csv(csv_path)
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if cl == 'ts_event': col_map[col] = 'timestamp'
        elif cl == 'open': col_map[col] = 'open'
        elif cl == 'high': col_map[col] = 'high'
        elif cl == 'low': col_map[col] = 'low'
        elif cl == 'close': col_map[col] = 'close'
        elif cl == 'volume': col_map[col] = 'volume'
    df = df.rename(columns=col_map)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df


def resample_to_15min(df, period=15):
    """Resample 1min OHLCV to N-min bars."""
    df = df.set_index('timestamp').sort_index()
    resampled = df.resample(f'{period}min').agg({
        'open': 'first', 'high': 'max', 'low': 'min',
        'close': 'last', 'volume': 'sum'
    }).dropna(subset=['close'])
    resampled = resampled.reset_index().rename(columns={'index': 'timestamp'})
    if 'timestamp' not in resampled.columns and resampled.index.name == 'timestamp':
        resampled = resampled.reset_index()
    return resampled


def load_and_prepare_data(stock, include_vol=False, include_cnn=False):
    """Load 1min splits from Google Drive, resample to 15min, compute features."""
    data_dir = os.path.join(cfg.DRIVE_BASE, f"{stock}_{cfg.FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))

    if not dfs:
        print(f"  [WARNING] Data not found for {stock} at {data_dir}")
        print(f"  → Generating synthetic data for testing")
        df = generate_synthetic_data(stock)
    else:
        df = pd.concat(dfs, ignore_index=True)
        df = df.sort_values('timestamp').reset_index(drop=True)
        df = resample_to_15min(df, cfg.RESAMPLE_PERIOD)
        print(f"  Loaded {stock}: {len(df)} bars ({cfg.RESAMPLE_PERIOD}min)")

    df['target_return'] = df['close'].pct_change(cfg.RETURN_HORIZON).shift(-cfg.RETURN_HORIZON)

    base_features = compute_technical_features(df)
    all_features = [base_features]

    if include_vol:
        print(f"  Running Vol model inference...")
        vol_features = run_vol_inference(
            df, cfg.VOL_MODEL_PATH, cfg.VOL_BLOCK_SIZE,
            cfg.VOL_SEQ_LEN, str(cfg.DEVICE))
        all_features.append(vol_features)

    if include_cnn:
        print(f"  Running CNN model inference...")
        cnn_features = run_cnn_inference(
            df, cfg.CNN_MODEL_PATH, cfg.CNN_WINDOW,
            cfg.CNN_N_FEATURES, str(cfg.DEVICE))
        all_features.append(cnn_features)

    features_df = pd.concat(all_features, axis=1)
    features_df = features_df.fillna(0)

    valid_mask = df['target_return'].notna()
    features_df = features_df[valid_mask].reset_index(drop=True)
    target_returns = df.loc[valid_mask, 'target_return'].values

    ret_std = np.std(target_returns)
    target_returns = np.clip(target_returns, -5 * ret_std, 5 * ret_std)

    return features_df, target_returns, list(features_df.columns)


def generate_synthetic_data(stock, n_bars=50000):
    """Generate synthetic OHLCV data for testing when real data unavailable."""
    np.random.seed(hash(stock) % 2**32)
    price = 100.0
    prices = [price]
    for _ in range(n_bars - 1):
        ret = np.random.normal(0, 0.001) + 0.0001 * (100 - price) / 100
        price *= (1 + ret)
        prices.append(price)
    prices = np.array(prices)
    noise = np.random.uniform(0.0005, 0.002, n_bars)
    df = pd.DataFrame({
        'timestamp': pd.date_range('2020-01-02 09:30', periods=n_bars, freq='15min'),
        'open': prices * (1 + np.random.normal(0, 0.0005, n_bars)),
        'high': prices * (1 + noise),
        'low': prices * (1 - noise),
        'close': prices,
        'volume': np.random.lognormal(10, 1, n_bars).astype(int)
    })
    return df


def normalize_features(train_feat, val_feat, test_feat):
    """Z-score normalization using train statistics only."""
    mean = np.nanmean(train_feat, axis=0)
    std = np.nanstd(train_feat, axis=0) + 1e-8
    return (train_feat - mean) / std, (val_feat - mean) / std, (test_feat - mean) / std


def split_data(features_df, returns, train_ratio=0.7, val_ratio=0.15):
    """Temporal split (no shuffling for time series)."""
    n = len(returns)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_feat = features_df.values[:train_end]
    val_feat = features_df.values[train_end:val_end]
    test_feat = features_df.values[val_end:]
    train_ret = returns[:train_end]
    val_ret = returns[train_end:val_end]
    test_ret = returns[val_end:]

    train_feat, val_feat, test_feat = normalize_features(train_feat, val_feat, test_feat)
    return (train_feat, train_ret), (val_feat, val_ret), (test_feat, test_ret)


# ============================================================
# TRAINING
# ============================================================

def train_one_epoch(model, loader, optimizer, loss_fn, mode='regression_only'):
    model.train()
    total_loss = 0
    n_batches = 0
    for x, y in loader:
        x, y = x.to(cfg.DEVICE), y.to(cfg.DEVICE).squeeze()
        optimizer.zero_grad()
        if mode == 'multi_task':
            pred_ret, pred_dir = model(x)
            pred_ret = pred_ret.squeeze()
            pred_dir = pred_dir.squeeze()
            loss, _, _ = loss_fn(pred_ret, pred_dir, y)
        else:
            pred_ret = model(x).squeeze()
            loss = loss_fn(pred_ret, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, loss_fn, mode='regression_only'):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    n_batches = 0
    for x, y in loader:
        x, y = x.to(cfg.DEVICE), y.to(cfg.DEVICE).squeeze()
        if mode == 'multi_task':
            pred_ret, pred_dir = model(x)
            pred_ret = pred_ret.squeeze()
            pred_dir = pred_dir.squeeze()
            loss, _, _ = loss_fn(pred_ret, pred_dir, y)
        else:
            pred_ret = model(x).squeeze()
            loss = loss_fn(pred_ret, y)
        total_loss += loss.item()
        all_preds.append(pred_ret.cpu().numpy())
        all_targets.append(y.cpu().numpy())
        n_batches += 1

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    pred_dir = (all_preds > 0).astype(int)
    actual_dir = (all_targets > 0).astype(int)
    dir_acc = np.mean(pred_dir == actual_dir)
    n_correct = np.sum(pred_dir == actual_dir)
    n_total = len(actual_dir)
    p_value = stats.binomtest(n_correct, n_total, 0.5, alternative='greater').pvalue
    mean_abs_pred = np.mean(np.abs(all_preds))
    pred_std = np.std(all_preds)

    return {
        'loss': total_loss / max(n_batches, 1),
        'dir_acc': dir_acc, 'p_value': p_value,
        'n_samples': n_total, 'n_correct': n_correct,
        'mean_abs_pred': mean_abs_pred, 'pred_std': pred_std,
    }


def train_model(model, train_loader, val_loader, loss_fn,
                mode='regression_only', epochs=50, patience=8, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, mode)
        val_metrics = evaluate(model, val_loader, loss_fn, mode)
        val_loss = val_metrics['loss']
        scheduler.step(val_loss)

        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_loss': val_loss, 'val_dir_acc': val_metrics['dir_acc'],
            'mean_abs_pred': val_metrics['mean_abs_pred'],
            'pred_std': val_metrics['pred_std'],
        })

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {train_loss:.6f} | "
                  f"Val: {val_loss:.6f} | DA: {val_metrics['dir_acc']:.4f} | "
                  f"|pred|: {val_metrics['mean_abs_pred']:.6f} | "
                  f"std(pred): {val_metrics['pred_std']:.6f}")

        if val_metrics['pred_std'] < 1e-7:
            print(f"  [WARNING] Epoch {epoch+1}: predictions collapsed to constant!")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(cfg.DEVICE)
    return history


@torch.no_grad()
def evaluate_by_magnitude(model, loader, mode='multi_task'):
    """Stratified DA analysis by |actual_return| magnitude."""
    model.eval()
    all_preds, all_targets = [], []
    for x, y in loader:
        x = x.to(cfg.DEVICE)
        if mode == 'multi_task':
            pred_ret, _ = model(x)
        else:
            pred_ret = model(x)
        all_preds.append(pred_ret.squeeze().cpu().numpy())
        all_targets.append(y.squeeze().cpu().numpy())

    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    abs_ret = np.abs(targets)

    print(f"\n  --- DA by Return Magnitude ---")
    print(f"  {'Group':<25} {'DA':>7} {'N':>7} {'p-value':>10} {'Sig':>5}")
    print(f"  {'-'*25} {'-'*7} {'-'*7} {'-'*10} {'-'*5}")

    results = {}
    da_all = np.mean((preds > 0) == (targets > 0))
    print(f"  {'All bars':<25} {da_all:>7.4f} {len(targets):>7}")

    groups = [
        ("Bottom 50% (noise)", 0, 50),
        ("50-75% (small moves)", 50, 75),
        ("75-90% (medium moves)", 75, 90),
        ("Top 10% (large moves)", 90, 100),
        ("Top 5% (very large)", 95, 100),
        ("Top 1% (extreme)", 99, 100),
    ]

    for name, pct_lo, pct_hi in groups:
        thr_lo = 0 if pct_lo == 0 else np.percentile(abs_ret, pct_lo)
        thr_hi = np.percentile(abs_ret, pct_hi) if pct_hi < 100 else abs_ret.max() + 1
        mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi) if pct_hi < 100 else (abs_ret >= thr_lo)
        n = mask.sum()
        if n >= 10:
            correct = ((preds[mask] > 0) == (targets[mask] > 0))
            da = np.mean(correct)
            n_correct = correct.sum()
            p_val = stats.binomtest(int(n_correct), int(n), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {name:<25} {da:>7.4f} {n:>7} {p_val:>10.6f} {sig:>5}")
            results[name] = {'da': da, 'n': int(n), 'p_value': p_val, 'threshold': float(thr_lo)}

    print(f"\n  --- DA for |return| > threshold ---")
    print(f"  {'Threshold':>12} {'DA':>7} {'N':>7} {'p-value':>10} {'Sig':>5}")
    for thr in [0.0005, 0.001, 0.002, 0.003, 0.005, 0.01]:
        mask = abs_ret > thr
        n = mask.sum()
        if n >= 20:
            correct = ((preds[mask] > 0) == (targets[mask] > 0))
            da = np.mean(correct)
            n_correct = correct.sum()
            p_val = stats.binomtest(int(n_correct), int(n), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {thr:>12.4f} {da:>7.4f} {n:>7} {p_val:>10.6f} {sig:>5}")
            results[f'thr_{thr}'] = {'da': da, 'n': int(n), 'p_value': p_val}

    return results


# ============================================================
# EXPERIMENT RUNNER
# ============================================================

def run_experiment(stock, step, include_vol=False, include_cnn=False):
    """Run a single experiment step for a given stock."""
    step_names = {
        1: "MSE Baseline (returns)",
        2: "GMADL Loss",
        3: "Multi-Task (GMADL + BCE)",
        4: "Multi-Task + Vol/CNN Features",
        5: "Attention LSTM + Vol/CNN",
        6: "Dual-Stream LSTM + Vol/CNN",
    }

    print(f"\n{'='*60}")
    print(f"Step {step}: {step_names[step]} | Stock: {stock}")
    print(f"{'='*60}")

    features_df, returns, feature_names = load_and_prepare_data(
        stock, include_vol=include_vol, include_cnn=include_cnn)
    n_features = len(feature_names)
    print(f"  Features: {n_features} | Samples: {len(returns)}")

    (train_f, train_r), (val_f, val_r), (test_f, test_r) = split_data(
        features_df, returns, cfg.TRAIN_RATIO, cfg.VAL_RATIO)
    print(f"  Train: {len(train_r)} | Val: {len(val_r)} | Test: {len(test_r)}")

    train_ds = ReturnPredictionDataset(train_f, train_r, cfg.SEQ_LEN)
    val_ds = ReturnPredictionDataset(val_f, val_r, cfg.SEQ_LEN)
    test_ds = ReturnPredictionDataset(test_f, test_r, cfg.SEQ_LEN)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

    mode = 'multi_task' if step >= 3 else 'regression_only'

    if step <= 4:
        model = LSTMReturnPredictor(
            n_features=n_features, hidden_size=cfg.HIDDEN_SIZE,
            num_layers=cfg.NUM_LAYERS, dropout=cfg.DROPOUT, mode=mode
        ).to(cfg.DEVICE)
    elif step == 5:
        model = AttentionLSTMPredictor(
            n_features=n_features, hidden_size=cfg.HIDDEN_SIZE,
            num_layers=cfg.NUM_LAYERS, dropout=cfg.DROPOUT, mode=mode
        ).to(cfg.DEVICE)
    elif step == 6:
        n_base = 16
        n_aux = n_features - n_base
        model = DualStreamLSTMPredictor(
            n_base_features=n_base, n_aux_features=n_aux,
            hidden_size=cfg.HIDDEN_SIZE, num_layers=cfg.NUM_LAYERS,
            dropout=cfg.DROPOUT, mode=mode
        ).to(cfg.DEVICE)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model params: {param_count:,} | Mode: {mode}")

    if step == 1:
        loss_fn = nn.MSELoss()
    elif step == 2:
        loss_fn = GMADLLoss(a=cfg.GMADL_A, b=cfg.GMADL_B)
    else:
        loss_fn = MultiTaskLoss(a=cfg.GMADL_A, b=cfg.GMADL_B, lambda_cls=cfg.LAMBDA_CLS)

    history = train_model(
        model, train_loader, val_loader, loss_fn,
        mode=mode, epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR)

    test_metrics = evaluate(model, test_loader, loss_fn, mode)

    print(f"\n  --- Test Results ---")
    print(f"  Directional Accuracy: {test_metrics['dir_acc']:.4f} "
          f"({test_metrics['n_correct']}/{test_metrics['n_samples']})")
    print(f"  P-value (vs 50%):     {test_metrics['p_value']:.6f}")
    print(f"  Mean |prediction|:    {test_metrics['mean_abs_pred']:.8f}")
    print(f"  Std(prediction):      {test_metrics['pred_std']:.8f}")

    sig = "***" if test_metrics['p_value'] < 0.001 else \
          "**" if test_metrics['p_value'] < 0.01 else \
          "*" if test_metrics['p_value'] < 0.05 else "ns"
    print(f"  Significance:         {sig}")

    if test_metrics['pred_std'] < 1e-7:
        print(f"  [COLLAPSED] Model outputs constant predictions!")

    if step >= 4 and test_metrics['pred_std'] > 1e-7:
        mag_results = evaluate_by_magnitude(model, test_loader, mode)
    else:
        mag_results = None

    return {
        'stock': stock, 'step': step, 'step_name': step_names[step],
        'n_features': n_features,
        'test_dir_acc': test_metrics['dir_acc'],
        'test_p_value': test_metrics['p_value'],
        'test_n_samples': test_metrics['n_samples'],
        'test_n_correct': test_metrics['n_correct'],
        'mean_abs_pred': test_metrics['mean_abs_pred'],
        'pred_std': test_metrics['pred_std'],
        'feature_names': feature_names,
        'magnitude_da': mag_results,
    }


# ============================================================
# MAIN: RUN ALL 6 STEPS
# ============================================================

def run_full_experiment(stocks=None):
    """Run all 6 experiment steps and compare results."""
    if stocks is None:
        stocks = cfg.STOCKS[:3]

    all_results = []
    for stock in stocks:
        print(f"\n{'#'*70}")
        print(f"# STOCK: {stock}")
        print(f"{'#'*70}")

        r1 = run_experiment(stock, step=1, include_vol=False, include_cnn=False)
        all_results.append(r1)
        r2 = run_experiment(stock, step=2, include_vol=False, include_cnn=False)
        all_results.append(r2)
        r3 = run_experiment(stock, step=3, include_vol=False, include_cnn=False)
        all_results.append(r3)
        r4 = run_experiment(stock, step=4, include_vol=True, include_cnn=True)
        all_results.append(r4)
        r5 = run_experiment(stock, step=5, include_vol=True, include_cnn=True)
        all_results.append(r5)
        r6 = run_experiment(stock, step=6, include_vol=True, include_cnn=True)
        all_results.append(r6)

    print(f"\n\n{'='*80}")
    print(f"SUMMARY OF ALL EXPERIMENTS")
    print(f"{'='*80}")
    print(f"{'Stock':<8} {'Step':<35} {'DA%':>8} {'N':>7} {'p-value':>10} {'Sig':>5} {'|pred|':>10} {'std':>10}")
    print(f"{'-'*8} {'-'*35} {'-'*8} {'-'*7} {'-'*10} {'-'*5} {'-'*10} {'-'*10}")

    for r in all_results:
        sig = "***" if r['test_p_value'] < 0.001 else \
              "**" if r['test_p_value'] < 0.01 else \
              "*" if r['test_p_value'] < 0.05 else "ns"
        collapsed = " [FLAT]" if r['pred_std'] < 1e-7 else ""
        print(f"{r['stock']:<8} {r['step_name']:<35} {r['test_dir_acc']:>7.4f} "
              f"{r['test_n_samples']:>7d} {r['test_p_value']:>10.6f} {sig:>5} "
              f"{r['mean_abs_pred']:>10.8f} {r['pred_std']:>10.8f}{collapsed}")

    print(f"\n--- Average by Step ---")
    results_df = pd.DataFrame(all_results)
    for step in [1, 2, 3, 4, 5, 6]:
        step_data = results_df[results_df['step'] == step]
        if len(step_data) > 0:
            avg_da = step_data['test_dir_acc'].mean()
            name = step_data['step_name'].iloc[0]
            n_sig = (step_data['test_p_value'] < 0.05).sum()
            print(f"  Step {step} ({name}): "
                  f"Avg DA = {avg_da:.4f} | "
                  f"Significant: {n_sig}/{len(step_data)}")

    return all_results


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    print(f"Device: {cfg.DEVICE}")
    print(f"GMADL params: a={cfg.GMADL_A}, b={cfg.GMADL_B}")
    print(f"Multi-task λ_cls={cfg.LAMBDA_CLS}")

    results = run_full_experiment(stocks=['AAPL', 'MSFT', 'SPY'])

    print("\n\nDone! Key things to check:")
    print("1. Step 1 (MSE): pred_std should be tiny → model predicting ~0 (copying)")
    print("2. Step 2 (GMADL): pred_std should be larger → model making directional bets")
    print("3. Step 3 (Multi-task): DA should improve over Step 2")
    print("4. Step 4 (+Vol/CNN): DA should improve over Step 3")
    print("5. Step 5 (Attention): Should focus on Vol/CNN features")
    print("6. Step 6 (Dual-Stream): Should preserve Vol/CNN signal")
    print("\nMagnitude analysis: Check if DA is higher for large-move bars")

Device: cuda
GMADL params: a=100.0, b=1.0
Multi-task λ_cls=0.3

######################################################################
# STOCK: AAPL
######################################################################

Step 1: MSE Baseline (returns) | Stock: AAPL
  Loaded AAPL: 39289 bars (15min)
  Features: 16 | Samples: 39288
  Train: 27501 | Val: 5893 | Test: 5894
  Model params: 215,425 | Mode: regression_only
  Epoch   1 | Train: 0.004204 | Val: 0.000033 | DA: 0.5066 | |pred|: 0.005122 | std(pred): 0.001086
  Epoch   5 | Train: 0.000021 | Val: 0.000007 | DA: 0.5066 | |pred|: 0.000849 | std(pred): 0.000151
  Epoch  10 | Train: 0.000010 | Val: 0.000006 | DA: 0.4934 | |pred|: 0.000106 | std(pred): 0.000022
  Epoch  15 | Train: 0.000008 | Val: 0.000006 | DA: 0.4934 | |pred|: 0.000076 | std(pred): 0.000005
  Epoch  20 | Train: 0.000009 | Val: 0.000006 | DA: 0.4934 | |pred|: 0.000099 | std(pred): 0.000005
  Epoch  25 | Train: 0.000008 | Val: 0.000006 | DA: 0.4934 | |pred|: 0.000039 | 

In [ ]:
"""
Direct Signal Diagnostic - Complete Runnable Version
====================================================
Paste this AFTER your GMADL_MultiTask_LSTM_FIXED.py cell in Colab.
It reuses all the functions already defined (Config, run_vol_inference, etc.)
"""

# ============================================================
# DIAGNOSTIC FUNCTIONS
# ============================================================

def evaluate_direct_cnn_signal(df, cnn_features, thresholds=[0.5, 0.6, 0.7, 0.8, 0.9]):
    """
    Directly evaluate: when CNN says 'bottom', does price actually go up?
    Bypasses LSTM entirely - tests raw CNN signal quality.
    """
    future_ret_1 = df['close'].pct_change().shift(-1)

    print(f"\n{'='*70}")
    print(f"DIRECT CNN SIGNAL EVALUATION (bypassing LSTM)")
    print(f"{'='*70}")

    # --- Bottom → price goes up? ---
    print(f"\n--- Bottom Detection → Price Goes UP? (1-bar) ---")
    print(f"{'Threshold':>10} {'N signals':>10} {'DA (up)':>8} {'p-value':>10} {'Sig':>5} {'Avg ret':>10}")

    for thr in thresholds:
        mask = cnn_features['cnn_prob_bottom'] > thr
        n = mask.sum()
        if n >= 10:
            rets = future_ret_1[mask].dropna()
            n_up = (rets > 0).sum()
            n_total = len(rets)
            da = n_up / n_total if n_total > 0 else 0
            avg_ret = rets.mean()
            p_val = stats.binomtest(int(n_up), int(n_total), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"{thr:>10.1f} {n_total:>10} {da:>8.4f} {p_val:>10.6f} {sig:>5} {avg_ret:>10.6f}")
        else:
            print(f"{thr:>10.1f} {n:>10}  (too few signals)")

    # --- Top → price goes down? ---
    print(f"\n--- Top Detection → Price Goes DOWN? (1-bar) ---")
    print(f"{'Threshold':>10} {'N signals':>10} {'DA (down)':>8} {'p-value':>10} {'Sig':>5} {'Avg ret':>10}")

    for thr in thresholds:
        mask = cnn_features['cnn_prob_top'] > thr
        n = mask.sum()
        if n >= 10:
            rets = future_ret_1[mask].dropna()
            n_down = (rets < 0).sum()
            n_total = len(rets)
            da = n_down / n_total if n_total > 0 else 0
            avg_ret = rets.mean()
            p_val = stats.binomtest(int(n_down), int(n_total), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"{thr:>10.1f} {n_total:>10} {da:>8.4f} {p_val:>10.6f} {sig:>5} {avg_ret:>10.6f}")
        else:
            print(f"{thr:>10.1f} {n:>10}  (too few signals)")

    # --- Net signal ---
    print(f"\n--- Net Signal: (prob_bottom - prob_top) as direction ---")
    print(f"{'Min |net|':>10} {'N signals':>10} {'DA':>8} {'p-value':>10} {'Sig':>5}")

    net = cnn_features['cnn_reversal_net']
    for min_net in [0.0, 0.1, 0.2, 0.3, 0.5]:
        mask = net.abs() > min_net
        n = mask.sum()
        if n >= 20:
            rets = future_ret_1[mask].dropna()
            signals = net[mask].reindex(rets.index)
            correct = ((signals > 0) & (rets > 0)) | ((signals < 0) & (rets < 0))
            da = correct.mean()
            n_correct = int(correct.sum())
            n_total = len(correct)
            p_val = stats.binomtest(n_correct, n_total, 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"{min_net:>10.1f} {n_total:>10} {da:>8.4f} {p_val:>10.6f} {sig:>5}")

    # --- KEY TEST: Multi-bar horizon (time-scale mismatch?) ---
    print(f"\n--- CNN Bottom Signal (>0.7) vs Multi-bar Future Return ---")
    print(f"  Testing if CNN signal works better at longer horizons")
    print(f"  {'Horizon':>10} {'N':>6} {'DA(up)':>8} {'p-value':>10} {'Sig':>5} {'Avg ret':>10}")

    mask = cnn_features['cnn_prob_bottom'] > 0.7
    for horizon in [1, 2, 3, 6, 12, 24]:
        fut = df['close'].pct_change(horizon).shift(-horizon)
        rets = fut[mask].dropna()
        if len(rets) >= 10:
            n_up = (rets > 0).sum()
            da = n_up / len(rets)
            p_val = stats.binomtest(int(n_up), len(rets), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {horizon:>10} {len(rets):>6} {da:>8.4f} {p_val:>10.6f} {sig:>5} {rets.mean():>10.6f}")
        else:
            print(f"  {horizon:>10} {len(rets):>6}  (too few)")

    print(f"\n--- CNN Top Signal (>0.7) vs Multi-bar Future Return ---")
    print(f"  {'Horizon':>10} {'N':>6} {'DA(dn)':>8} {'p-value':>10} {'Sig':>5} {'Avg ret':>10}")

    mask = cnn_features['cnn_prob_top'] > 0.7
    for horizon in [1, 2, 3, 6, 12, 24]:
        fut = df['close'].pct_change(horizon).shift(-horizon)
        rets = fut[mask].dropna()
        if len(rets) >= 10:
            n_down = (rets < 0).sum()
            da = n_down / len(rets)
            p_val = stats.binomtest(int(n_down), len(rets), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {horizon:>10} {len(rets):>6} {da:>8.4f} {p_val:>10.6f} {sig:>5} {rets.mean():>10.6f}")
        else:
            print(f"  {horizon:>10} {len(rets):>6}  (too few)")


def evaluate_vol_gated_signal(df, vol_features, cnn_features,
                               vol_threshold_pct=75, cnn_thresholds=[0.5, 0.6, 0.7, 0.8]):
    """
    Strategy: Only trade when Vol predicts high volatility AND CNN gives direction.
    """
    future_ret = df['close'].pct_change().shift(-1)

    print(f"\n{'='*70}")
    print(f"VOL-GATED CNN STRATEGY")
    print(f"{'='*70}")

    vol_pred = vol_features['vol_pred_rv']
    vol_high_thr = vol_pred.quantile(vol_threshold_pct / 100)
    vol_high = vol_pred > vol_high_thr

    print(f"\n  Vol threshold: {vol_threshold_pct}th percentile = {vol_high_thr:.6f}")
    print(f"  Bars with high vol predicted: {vol_high.sum()}")

    for cnn_thr in cnn_thresholds:
        buy_signal = vol_high & (cnn_features['cnn_prob_bottom'] > cnn_thr)
        sell_signal = vol_high & (cnn_features['cnn_prob_top'] > cnn_thr)

        buy_rets = future_ret[buy_signal].dropna()
        sell_rets = future_ret[sell_signal].dropna()

        print(f"\n  CNN threshold: {cnn_thr}")

        if len(buy_rets) >= 5:
            n_up = (buy_rets > 0).sum()
            da = n_up / len(buy_rets)
            p_str = ""
            if len(buy_rets) >= 10:
                p_val = stats.binomtest(int(n_up), len(buy_rets), 0.5, alternative='greater').pvalue
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                p_str = f", p={p_val:.4f} {sig}"
            print(f"    BUY signals:  N={len(buy_rets):>5}, DA(up)={da:.4f}, avg_ret={buy_rets.mean():.6f}{p_str}")
        else:
            print(f"    BUY signals:  N={len(buy_rets):>5} (too few)")

        if len(sell_rets) >= 5:
            n_down = (sell_rets < 0).sum()
            da = n_down / len(sell_rets)
            p_str = ""
            if len(sell_rets) >= 10:
                p_val = stats.binomtest(int(n_down), len(sell_rets), 0.5, alternative='greater').pvalue
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
                p_str = f", p={p_val:.4f} {sig}"
            print(f"    SELL signals: N={len(sell_rets):>5}, DA(dn)={da:.4f}, avg_ret={sell_rets.mean():.6f}{p_str}")
        else:
            print(f"    SELL signals: N={len(sell_rets):>5} (too few)")


# ============================================================
# RUN DIAGNOSTIC FOR ALL 3 STOCKS
# ============================================================

print(f"{'#'*70}")
print(f"# CNN/VOL DIRECT SIGNAL DIAGNOSTIC")
print(f"{'#'*70}")

for stock in ['AAPL', 'MSFT', 'SPY']:
    print(f"\n\n{'#'*70}")
    print(f"# STOCK: {stock}")
    print(f"{'#'*70}")

    # Load data (reusing existing functions from GMADL cell)
    data_dir = os.path.join(cfg.DRIVE_BASE, f"{stock}_{cfg.FREQ_RAW}")
    dfs = []
    for split in ['train', 'test']:
        fpath = os.path.join(data_dir, f"{split}.csv")
        if os.path.exists(fpath):
            dfs.append(load_split_csv(fpath))

    if not dfs:
        print(f"  Data not found for {stock}, skipping")
        continue

    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = resample_to_15min(df, cfg.RESAMPLE_PERIOD)
    print(f"  Loaded {stock}: {len(df)} bars ({cfg.RESAMPLE_PERIOD}min)")

    # Use only TEST portion (last 15%)
    n = len(df)
    test_start = int(n * (cfg.TRAIN_RATIO + cfg.VAL_RATIO))
    df_test = df.iloc[test_start:].reset_index(drop=True)
    print(f"  Test portion: {len(df_test)} bars (from bar {test_start})")

    # Run Vol inference on test data
    print(f"  Running Vol model inference...")
    vol_features = run_vol_inference(
        df_test, cfg.VOL_MODEL_PATH, cfg.VOL_BLOCK_SIZE,
        cfg.VOL_SEQ_LEN, str(cfg.DEVICE))

    # Run CNN inference on test data
    print(f"  Running CNN model inference...")
    cnn_features = run_cnn_inference(
        df_test, cfg.CNN_MODEL_PATH, cfg.CNN_WINDOW,
        cfg.CNN_N_FEATURES, str(cfg.DEVICE))

    # Run diagnostics
    evaluate_direct_cnn_signal(df_test, cnn_features)
    evaluate_vol_gated_signal(df_test, vol_features, cnn_features)

print(f"\n\n{'='*70}")
print("INTERPRETATION:")
print("="*70)
print("1. If CNN bottom/top DA ≈ 50% at horizon=1 but > 50% at horizon=6:")
print("   → Time-scale mismatch! Change GMADL RETURN_HORIZON from 1 to 6")
print("")
print("2. If CNN DA ≈ 50% at ALL horizons:")
print("   → CNN turning point detection doesn't translate to direction signal")
print("   → Use CNN only for volatility gating, not direction")
print("")
print("3. If Vol-gated CNN shows higher DA than CNN alone:")
print("   → Correct integration: Vol filters WHEN to trade, CNN gives direction")
print("   → Implement as rule-based strategy, not LSTM features")

######################################################################
# CNN/VOL DIRECT SIGNAL DIAGNOSTIC
######################################################################


######################################################################
# STOCK: AAPL
######################################################################
  Loaded AAPL: 39289 bars (15min)
  Test portion: 5894 bars (from bar 33395)
  Running Vol model inference...
  Running CNN model inference...
  CNN prob_bottom: mean=0.345, std=0.188, range=[0.039, 0.969]
  CNN prob_top:    mean=0.268, std=0.132, range=[0.040, 0.778]

DIRECT CNN SIGNAL EVALUATION (bypassing LSTM)

--- Bottom Detection → Price Goes UP? (1-bar) ---
 Threshold  N signals  DA (up)    p-value   Sig    Avg ret
       0.5       1146   0.4930   0.692219    ns   0.000056
       0.6        614   0.5114   0.299936    ns   0.000166
       0.7        336   0.5000   0.521748    ns  -0.000016
       0.8        181   0.4862   0.672133    ns  -0.000063
   

In [ ]:
"""
Magnitude-Stratified DA Analysis
=================================
Quick script: Train Step 4 model (GMADL+BCE+Vol/CNN) for each stock,
then break down DA by |return| magnitude to test the hypothesis
that DA is much higher for large-move bars.

Run after Train_Save_VolCNN.py has saved model weights.
"""

import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy import stats

# ============================================================
# Copy essential classes from GMADL code
# ============================================================

# --- Import everything from GMADL cell ---
# If running in same Colab notebook after GMADL cell, these are already defined.
# Otherwise, paste the GMADL cell code first.

# This script assumes the following are available from GMADL cell:
#   Config (cfg), LSTMReturnPredictor, MultiTaskLoss, GMADLLoss,
#   ReturnPredictionDataset, load_and_prepare_data, split_data,
#   normalize_features, train_model, evaluate, VolLSTM, CNNDualModel

# ============================================================
# MAGNITUDE ANALYSIS
# ============================================================

@torch.no_grad()
def magnitude_analysis(model, test_loader, mode, stock_name, step_name, device):
    """Detailed DA breakdown by return magnitude."""
    model.eval()
    all_preds, all_targets = [], []

    for x, y in test_loader:
        x = x.to(device)
        if mode == 'multi_task':
            pred_ret, _ = model(x)
        else:
            pred_ret = model(x)
        all_preds.append(pred_ret.squeeze().cpu().numpy())
        all_targets.append(y.squeeze().cpu().numpy())

    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    abs_ret = np.abs(targets)

    print(f"\n{'='*70}")
    print(f"  MAGNITUDE ANALYSIS: {stock_name} | {step_name}")
    print(f"{'='*70}")

    # Overall stats
    da_all = np.mean((preds > 0) == (targets > 0))
    n_total = len(targets)
    print(f"  Overall: DA={da_all:.4f} (N={n_total})")
    print(f"  Return stats: mean={targets.mean():.6f}, std={targets.std():.6f}")
    print(f"  |Return| percentiles: 25%={np.percentile(abs_ret,25):.6f}, "
          f"50%={np.percentile(abs_ret,50):.6f}, "
          f"75%={np.percentile(abs_ret,75):.6f}, "
          f"90%={np.percentile(abs_ret,90):.6f}, "
          f"95%={np.percentile(abs_ret,95):.6f}")

    # --- Part 1: Percentile groups ---
    print(f"\n  --- DA by Percentile Group ---")
    print(f"  {'Group':<30} {'|ret| range':<25} {'DA':>7} {'N':>6} {'p-value':>10} {'Sig':>4}")
    print(f"  {'-'*30} {'-'*25} {'-'*7} {'-'*6} {'-'*10} {'-'*4}")

    pct_groups = [
        ("Bottom 25% (flat)", 0, 25),
        ("25-50%", 25, 50),
        ("50-75%", 50, 75),
        ("75-90% (medium)", 75, 90),
        ("90-95% (large)", 90, 95),
        ("Top 5% (very large)", 95, 100),
        ("Top 2.5% (extreme)", 97.5, 100),
        ("Top 1% (extreme)", 99, 100),
    ]

    results = {}
    for name, pct_lo, pct_hi in pct_groups:
        thr_lo = np.percentile(abs_ret, pct_lo) if pct_lo > 0 else 0
        thr_hi = np.percentile(abs_ret, pct_hi) if pct_hi < 100 else abs_ret.max() + 1

        if pct_hi == 100:
            mask = abs_ret >= thr_lo
        else:
            mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi)

        n = mask.sum()
        if n >= 10:
            correct = (preds[mask] > 0) == (targets[mask] > 0)
            da = np.mean(correct)
            n_correct = int(correct.sum())
            p_val = stats.binomtest(n_correct, int(n), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            range_str = f"[{thr_lo:.5f}, {thr_hi:.5f})"
            print(f"  {name:<30} {range_str:<25} {da:>7.4f} {n:>6} {p_val:>10.6f} {sig:>4}")
            results[name] = {'da': da, 'n': int(n), 'p': p_val}

    # --- Part 2: Fixed thresholds ---
    print(f"\n  --- DA for |return| > threshold ---")
    print(f"  {'Threshold':<15} {'DA':>7} {'N':>6} {'% of data':>10} {'p-value':>10} {'Sig':>4}")
    print(f"  {'-'*15} {'-'*7} {'-'*6} {'-'*10} {'-'*10} {'-'*4}")

    thresholds = [0, 0.0002, 0.0005, 0.001, 0.0015, 0.002, 0.003, 0.005, 0.008, 0.01, 0.015, 0.02]
    for thr in thresholds:
        mask = abs_ret > thr
        n = mask.sum()
        if n >= 20:
            correct = (preds[mask] > 0) == (targets[mask] > 0)
            da = np.mean(correct)
            n_correct = int(correct.sum())
            pct_data = n / n_total * 100
            p_val = stats.binomtest(n_correct, int(n), 0.5, alternative='greater').pvalue
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"  {thr:<15.4f} {da:>7.4f} {n:>6} {pct_data:>9.1f}% {p_val:>10.6f} {sig:>4}")

    # --- Part 3: Up vs Down ---
    print(f"\n  --- DA by Direction ---")
    up_mask = targets > 0
    down_mask = targets < 0
    da_up = np.mean(preds[up_mask] > 0) if up_mask.sum() > 0 else 0
    da_down = np.mean(preds[down_mask] < 0) if down_mask.sum() > 0 else 0
    print(f"  Up moves:   DA={da_up:.4f} (correctly predicted up, N={up_mask.sum()})")
    print(f"  Down moves: DA={da_down:.4f} (correctly predicted down, N={down_mask.sum()})")

    # Large up vs large down
    big_up = (targets > np.percentile(targets, 90))
    big_down = (targets < np.percentile(targets, 10))
    if big_up.sum() > 10:
        da_big_up = np.mean(preds[big_up] > 0)
        print(f"  Big up (top 10%):    DA={da_big_up:.4f} (N={big_up.sum()})")
    if big_down.sum() > 10:
        da_big_down = np.mean(preds[big_down] < 0)
        print(f"  Big down (bot 10%):  DA={da_big_down:.4f} (N={big_down.sum()})")

    return results


# ============================================================
# MAIN: Run Step 4 + Magnitude Analysis for 3 stocks
# ============================================================

def run_magnitude_experiment(stocks=['AAPL', 'MSFT', 'SPY']):
    """Train Step 4 models and run magnitude analysis."""

    all_mag_results = {}

    for stock in stocks:
        print(f"\n{'#'*70}")
        print(f"# {stock}: Training Step 4 + Magnitude Analysis")
        print(f"{'#'*70}")

        # Load data with Vol/CNN
        features_df, returns, feature_names = load_and_prepare_data(
            stock, include_vol=True, include_cnn=True
        )
        n_features = len(feature_names)
        print(f"  Features: {n_features} | Samples: {len(returns)}")

        # Split
        (train_f, train_r), (val_f, val_r), (test_f, test_r) = split_data(
            features_df, returns, cfg.TRAIN_RATIO, cfg.VAL_RATIO
        )

        # Datasets
        train_ds = ReturnPredictionDataset(train_f, train_r, cfg.SEQ_LEN)
        val_ds = ReturnPredictionDataset(val_f, val_r, cfg.SEQ_LEN)
        test_ds = ReturnPredictionDataset(test_f, test_r, cfg.SEQ_LEN)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

        # Build model (Step 4: vanilla LSTM + multi-task)
        mode = 'multi_task'
        model = LSTMReturnPredictor(
            n_features=n_features,
            hidden_size=cfg.HIDDEN_SIZE,
            num_layers=cfg.NUM_LAYERS,
            dropout=cfg.DROPOUT,
            mode=mode
        ).to(cfg.DEVICE)

        loss_fn = MultiTaskLoss(a=cfg.GMADL_A, b=cfg.GMADL_B, lambda_cls=cfg.LAMBDA_CLS)

        # Train
        print(f"  Training...")
        history = train_model(
            model, train_loader, val_loader, loss_fn,
            mode=mode, epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR
        )

        # Quick overall test
        test_metrics = evaluate(model, test_loader, loss_fn, mode)
        print(f"\n  Overall Test DA: {test_metrics['dir_acc']:.4f} "
              f"(p={test_metrics['p_value']:.6f})")

        # Magnitude analysis
        mag = magnitude_analysis(
            model, test_loader, mode, stock, "Step 4 (GMADL+BCE+Vol/CNN)", cfg.DEVICE
        )
        all_mag_results[stock] = mag

    # --- Cross-stock summary ---
    print(f"\n\n{'='*70}")
    print(f"  CROSS-STOCK MAGNITUDE SUMMARY")
    print(f"{'='*70}")

    common_groups = ["Bottom 25% (flat)", "25-50%", "50-75%",
                     "75-90% (medium)", "90-95% (large)", "Top 5% (very large)"]

    print(f"\n  {'Group':<30}", end="")
    for stock in stocks:
        print(f" {stock:>10}", end="")
    print(f" {'Average':>10}")
    print(f"  {'-'*30}", end="")
    for _ in stocks:
        print(f" {'-'*10}", end="")
    print(f" {'-'*10}")

    for group in common_groups:
        print(f"  {group:<30}", end="")
        das = []
        for stock in stocks:
            if stock in all_mag_results and group in all_mag_results[stock]:
                da = all_mag_results[stock][group]['da']
                das.append(da)
                print(f" {da:>9.4f}", end="")
            else:
                print(f" {'N/A':>10}", end="")
        if das:
            print(f" {np.mean(das):>9.4f}")
        else:
            print()

    return all_mag_results


# Run it
print("="*70)
print("  MAGNITUDE-STRATIFIED DA ANALYSIS")
print("  Hypothesis: DA >> 50% for large moves, ~50% for noise bars")
print("="*70)

mag_results = run_magnitude_experiment(stocks=['AAPL', 'MSFT', 'SPY'])

  MAGNITUDE-STRATIFIED DA ANALYSIS
  Hypothesis: DA >> 50% for large moves, ~50% for noise bars

######################################################################
# AAPL: Training Step 4 + Magnitude Analysis
######################################################################
  Loaded AAPL: 39289 bars (15min)
  Running Vol model inference...
  Running CNN model inference...
  CNN prob_bottom: mean=0.337, std=0.189, range=[0.032, 0.992]
  CNN prob_top:    mean=0.285, std=0.161, range=[0.026, 0.976]
  Features: 24 | Samples: 39288
  Training...
  Epoch   1 | Train: 0.208454 | Val: 0.207945 | DA: 0.4941 | |pred|: 0.892557 | std(pred): 0.344062
  Epoch   5 | Train: 0.207369 | Val: 0.207813 | DA: 0.5030 | |pred|: 3.969439 | std(pred): 1.955368
  Epoch  10 | Train: 0.206744 | Val: 0.207500 | DA: 0.5087 | |pred|: 2.920072 | std(pred): 3.020880
  Epoch  15 | Train: 0.206337 | Val: 0.207582 | DA: 0.5150 | |pred|: 3.712937 | std(pred): 4.240753
  Early stopping at epoch 19

  Overall Test

In [ ]:
"""
Debiased Magnitude Analysis
=============================
Check whether the magnitude-DA relationship is genuine or inflated by
directional bias (e.g., model predicting "always up").

Key tests:
1. Within large moves, separately check DA for up-moves and down-moves
2. If DA is high for BOTH directions among large moves → genuine signal
3. If DA is only high for one direction → bias artifact
4. Compare model DA vs "always up" baseline at each magnitude level
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from scipy import stats

@torch.no_grad()
def debiased_magnitude_analysis(model, test_loader, mode, stock_name, device):
    """Full debiased analysis."""
    model.eval()
    all_preds, all_targets = [], []

    for x, y in test_loader:
        x = x.to(device)
        if mode == 'multi_task':
            pred_ret, _ = model(x)
        else:
            pred_ret = model(x)
        all_preds.append(pred_ret.squeeze().cpu().numpy())
        all_targets.append(y.squeeze().cpu().numpy())

    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    abs_ret = np.abs(targets)

    pred_up_pct = np.mean(preds > 0) * 100
    actual_up_pct = np.mean(targets > 0) * 100

    print(f"\n{'='*75}")
    print(f"  DEBIASED ANALYSIS: {stock_name}")
    print(f"{'='*75}")
    print(f"  Prediction bias: {pred_up_pct:.1f}% predicted UP vs {actual_up_pct:.1f}% actual UP")
    print(f"  → Model {'bullish bias' if pred_up_pct > 60 else 'bearish bias' if pred_up_pct < 40 else 'balanced'}")

    # -------------------------------------------------------
    # TEST 1: DA by magnitude, SEPARATELY for up and down
    # -------------------------------------------------------
    print(f"\n  --- DA by Magnitude × Direction ---")
    print(f"  {'Group':<25} {'All DA':>7} {'Up DA':>7} {'Up N':>6} {'Down DA':>8} {'Down N':>7} {'Balanced':>9}")
    print(f"  {'-'*25} {'-'*7} {'-'*7} {'-'*6} {'-'*8} {'-'*7} {'-'*9}")

    groups = [
        ("All bars", 0, 100),
        ("Bottom 50% (flat)", 0, 50),
        ("50-75%", 50, 75),
        ("75-90% (medium)", 75, 90),
        ("Top 10% (large)", 90, 100),
        ("Top 5% (very large)", 95, 100),
        ("Top 1% (extreme)", 99, 100),
    ]

    for name, pct_lo, pct_hi in groups:
        thr_lo = np.percentile(abs_ret, pct_lo) if pct_lo > 0 else 0

        if pct_hi == 100:
            mask = abs_ret >= thr_lo
        else:
            thr_hi = np.percentile(abs_ret, pct_hi)
            mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi)

        n = mask.sum()
        if n < 20:
            continue

        p_sub = preds[mask]
        t_sub = targets[mask]

        # Overall DA
        da_all = np.mean((p_sub > 0) == (t_sub > 0))

        # DA for actual up moves
        up_mask = t_sub > 0
        n_up = up_mask.sum()
        da_up = np.mean(p_sub[up_mask] > 0) if n_up > 5 else float('nan')

        # DA for actual down moves
        down_mask = t_sub < 0
        n_down = down_mask.sum()
        da_down = np.mean(p_sub[down_mask] < 0) if n_down > 5 else float('nan')

        # Balanced DA = (DA_up + DA_down) / 2
        # This removes the effect of directional bias
        if n_up > 5 and n_down > 5:
            balanced_da = (da_up + da_down) / 2
        else:
            balanced_da = float('nan')

        print(f"  {name:<25} {da_all:>7.4f} {da_up:>7.4f} {n_up:>6} {da_down:>8.4f} {n_down:>7} {balanced_da:>9.4f}")

    # -------------------------------------------------------
    # TEST 2: "Always Up" baseline comparison
    # -------------------------------------------------------
    print(f"\n  --- Model DA vs 'Always Up' Baseline ---")
    print(f"  {'Group':<25} {'Model DA':>9} {'AlwaysUp':>9} {'Delta':>7} {'Signal?':>8}")
    print(f"  {'-'*25} {'-'*9} {'-'*9} {'-'*7} {'-'*8}")

    for name, pct_lo, pct_hi in groups:
        thr_lo = np.percentile(abs_ret, pct_lo) if pct_lo > 0 else 0

        if pct_hi == 100:
            mask = abs_ret >= thr_lo
        else:
            thr_hi = np.percentile(abs_ret, pct_hi)
            mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi)

        n = mask.sum()
        if n < 20:
            continue

        t_sub = targets[mask]
        p_sub = preds[mask]

        da_model = np.mean((p_sub > 0) == (t_sub > 0))
        da_always_up = np.mean(t_sub > 0)  # "always predict up" DA
        delta = da_model - da_always_up
        signal = "YES" if delta > 0.01 else "marginal" if delta > 0 else "NO"

        print(f"  {name:<25} {da_model:>9.4f} {da_always_up:>9.4f} {delta:>+7.4f} {signal:>8}")

    # -------------------------------------------------------
    # TEST 3: Prediction flip rate by magnitude
    # -------------------------------------------------------
    print(f"\n  --- Prediction Distribution by Magnitude ---")
    print(f"  {'Group':<25} {'%Pred UP':>9} {'%Pred DOWN':>11} {'%Actual UP':>11}")
    print(f"  {'-'*25} {'-'*9} {'-'*11} {'-'*11}")

    for name, pct_lo, pct_hi in groups:
        thr_lo = np.percentile(abs_ret, pct_lo) if pct_lo > 0 else 0

        if pct_hi == 100:
            mask = abs_ret >= thr_lo
        else:
            thr_hi = np.percentile(abs_ret, pct_hi)
            mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi)

        n = mask.sum()
        if n < 20:
            continue

        p_sub = preds[mask]
        t_sub = targets[mask]

        pct_pred_up = np.mean(p_sub > 0) * 100
        pct_pred_down = np.mean(p_sub <= 0) * 100
        pct_actual_up = np.mean(t_sub > 0) * 100

        print(f"  {name:<25} {pct_pred_up:>8.1f}% {pct_pred_down:>10.1f}% {pct_actual_up:>10.1f}%")

    # -------------------------------------------------------
    # TEST 4: Balanced DA with binomial test
    # -------------------------------------------------------
    print(f"\n  --- Balanced DA (bias-corrected) with significance ---")
    print(f"  {'Group':<25} {'Bal. DA':>8} {'N_eff':>6} {'p-value':>10} {'Sig':>4}")
    print(f"  {'-'*25} {'-'*8} {'-'*6} {'-'*10} {'-'*4}")

    for name, pct_lo, pct_hi in groups:
        thr_lo = np.percentile(abs_ret, pct_lo) if pct_lo > 0 else 0

        if pct_hi == 100:
            mask = abs_ret >= thr_lo
        else:
            thr_hi = np.percentile(abs_ret, pct_hi)
            mask = (abs_ret >= thr_lo) & (abs_ret < thr_hi)

        n = mask.sum()
        if n < 20:
            continue

        p_sub = preds[mask]
        t_sub = targets[mask]

        up_mask = t_sub > 0
        down_mask = t_sub < 0
        n_up = up_mask.sum()
        n_down = down_mask.sum()

        if n_up < 10 or n_down < 10:
            continue

        # Correct predictions in each direction
        correct_up = np.sum(p_sub[up_mask] > 0)
        correct_down = np.sum(p_sub[down_mask] < 0)
        total_correct = correct_up + correct_down
        n_eff = n_up + n_down

        balanced_da = total_correct / n_eff

        # Binomial test on balanced correct count
        p_val = stats.binomtest(int(total_correct), int(n_eff), 0.5,
                                alternative='greater').pvalue
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

        print(f"  {name:<25} {balanced_da:>8.4f} {n_eff:>6} {p_val:>10.6f} {sig:>4}")


# ============================================================
# MAIN
# ============================================================

def run_debiased(stocks=['AAPL', 'MSFT', 'SPY']):
    for stock in stocks:
        print(f"\n{'#'*75}")
        print(f"# {stock}")
        print(f"{'#'*75}")

        # Load data
        features_df, returns, feature_names = load_and_prepare_data(
            stock, include_vol=True, include_cnn=True
        )
        n_features = len(feature_names)

        # Split
        (train_f, train_r), (val_f, val_r), (test_f, test_r) = split_data(
            features_df, returns, cfg.TRAIN_RATIO, cfg.VAL_RATIO
        )

        # Datasets
        train_ds = ReturnPredictionDataset(train_f, train_r, cfg.SEQ_LEN)
        val_ds = ReturnPredictionDataset(val_f, val_r, cfg.SEQ_LEN)
        test_ds = ReturnPredictionDataset(test_f, test_r, cfg.SEQ_LEN)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

        # Train Step 4
        mode = 'multi_task'
        model = LSTMReturnPredictor(
            n_features=n_features,
            hidden_size=cfg.HIDDEN_SIZE,
            num_layers=cfg.NUM_LAYERS,
            dropout=cfg.DROPOUT,
            mode=mode
        ).to(cfg.DEVICE)

        loss_fn = MultiTaskLoss(a=cfg.GMADL_A, b=cfg.GMADL_B, lambda_cls=cfg.LAMBDA_CLS)

        print(f"  Training Step 4...")
        train_model(model, train_loader, val_loader, loss_fn,
                    mode=mode, epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR)

        # Debiased analysis
        debiased_magnitude_analysis(model, test_loader, mode, stock, cfg.DEVICE)


run_debiased()


###########################################################################
# AAPL
###########################################################################
  Loaded AAPL: 39289 bars (15min)
  Running Vol model inference...
  Running CNN model inference...
  Training Step 4...
  Epoch   1 | Train: 0.208780 | Val: 0.208205 | DA: 0.4934 | |pred|: 0.358706 | std(pred): 0.167418
  Epoch   5 | Train: 0.207445 | Val: 0.207796 | DA: 0.5076 | |pred|: 2.527441 | std(pred): 1.691802
  Epoch  10 | Train: 0.207082 | Val: 0.207293 | DA: 0.5083 | |pred|: 4.219643 | std(pred): 3.466838
  Epoch  15 | Train: 0.206342 | Val: 0.207657 | DA: 0.5234 | |pred|: 3.830261 | std(pred): 4.594134
  Early stopping at epoch 18

  DEBIASED ANALYSIS: AAPL
  Prediction bias: 85.9% predicted UP vs 49.9% actual UP
  → Model bullish bias

  --- DA by Magnitude × Direction ---
  Group                      All DA   Up DA   Up N  Down DA  Down N  Balanced
  ------------------------- ------- ------- ------ -------- ------

In [ ]:
"""
LSTM Prediction Lag Diagnostic
================================
Check if the GMADL LSTM (Step 4) has prediction lag issues:

1. Cross-correlation: pred[t] vs actual[t], actual[t-1], actual[t-2]...
   → If corr(pred[t], actual[t-1]) > corr(pred[t], actual[t]), model is lagging

2. Scatter plots: pred vs actual, pred vs actual_shifted

3. Direction change analysis: when actual direction flips,
   does prediction flip at same time or 1-2 bars later?

4. Autocorrelation of predictions vs autocorrelation of actuals
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy import stats

@torch.no_grad()
def get_predictions(model, test_loader, mode, device):
    """Extract all predictions and targets from test set."""
    model.eval()
    all_preds, all_targets = [], []
    for x, y in test_loader:
        x = x.to(device)
        if mode == 'multi_task':
            pred_ret, _ = model(x)
        else:
            pred_ret = model(x)
        all_preds.append(pred_ret.squeeze().cpu().numpy())
        all_targets.append(y.squeeze().cpu().numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)


def lag_diagnostic(preds, targets, stock_name):
    """Full lag analysis."""

    print(f"\n{'='*70}")
    print(f"  LAG DIAGNOSTIC: {stock_name}")
    print(f"{'='*70}")

    # -------------------------------------------------------
    # TEST 1: Cross-correlation at different lags
    # -------------------------------------------------------
    # corr(pred[t], actual[t-k]) for k = -3 to +3
    # If model is lagging, corr at k=1 (actual shifted back) will be higher

    print(f"\n  --- Cross-Correlation: pred[t] vs actual[t-k] ---")
    print(f"  {'Lag k':<10} {'Correlation':>12} {'Interpretation':<30}")
    print(f"  {'-'*10} {'-'*12} {'-'*30}")

    max_corr = -1
    max_lag = 0

    for k in range(-5, 6):
        if k >= 0:
            p = preds[k:]
            a = targets[:len(targets)-k] if k > 0 else targets
        else:
            p = preds[:len(preds)+k]
            a = targets[-k:]

        if len(p) < 100:
            continue

        corr, p_val = stats.pearsonr(p, a)

        if k == 0:
            interp = "← CURRENT (should be highest)"
        elif k == 1:
            interp = "← pred correlates with PAST actual"
        elif k == -1:
            interp = "← pred correlates with FUTURE actual"
        else:
            interp = ""

        marker = " ***" if abs(corr) > max_corr else ""
        if abs(corr) > max_corr:
            max_corr = abs(corr)
            max_lag = k

        print(f"  k={k:<6} {corr:>12.6f}{marker} {interp}")

    if max_lag == 0:
        print(f"\n  ✓ Peak correlation at k=0 → NO LAG detected")
    elif max_lag > 0:
        print(f"\n  ⚠ Peak correlation at k={max_lag} → MODEL LAGS by {max_lag} bar(s)!")
        print(f"    Predictions are copying past returns, not predicting future")
    else:
        print(f"\n  ? Peak correlation at k={max_lag} → model leads actual (unusual)")

    # -------------------------------------------------------
    # TEST 2: Direction agreement timing
    # -------------------------------------------------------
    print(f"\n  --- Direction Change Analysis ---")

    # Find where actual direction changes
    actual_dir = (targets > 0).astype(int)
    pred_dir = (preds > 0).astype(int)

    # Direction flips in actual
    actual_flips = np.where(np.diff(actual_dir) != 0)[0]
    n_flips = len(actual_flips)

    if n_flips > 10:
        # At each actual flip, check if pred also flips at same time, 1 bar later, etc
        lag_counts = {-2: 0, -1: 0, 0: 0, 1: 0, 2: 0, 'none': 0}

        for flip_idx in actual_flips:
            found = False
            for lag in [-2, -1, 0, 1, 2]:
                check_idx = flip_idx + lag
                if 0 < check_idx < len(pred_dir) - 1:
                    if np.diff(pred_dir)[check_idx] != 0:
                        lag_counts[lag] += 1
                        found = True
                        break
            if not found:
                lag_counts['none'] += 1

        print(f"  Actual direction flips: {n_flips}")
        print(f"  Prediction flip timing relative to actual flip:")
        for lag in [-2, -1, 0, 1, 2]:
            pct = lag_counts[lag] / n_flips * 100
            label = "pred flips BEFORE actual" if lag < 0 else \
                    "pred flips SAME time" if lag == 0 else \
                    "pred flips AFTER actual (LAGGING)"
            print(f"    Lag {lag:+d}: {lag_counts[lag]:>5} ({pct:>5.1f}%) {label}")
        pct_none = lag_counts['none'] / n_flips * 100
        print(f"    None:  {lag_counts['none']:>5} ({pct_none:>5.1f}%) pred doesn't flip nearby")

    # -------------------------------------------------------
    # TEST 3: Autocorrelation comparison
    # -------------------------------------------------------
    print(f"\n  --- Autocorrelation ---")
    print(f"  {'Lag':<6} {'Actual AC':>12} {'Pred AC':>12} {'Interpretation':<30}")
    print(f"  {'-'*6} {'-'*12} {'-'*12} {'-'*30}")

    for lag in [1, 2, 3, 5, 10]:
        ac_actual = np.corrcoef(targets[lag:], targets[:-lag])[0, 1]
        ac_pred = np.corrcoef(preds[lag:], preds[:-lag])[0, 1]

        interp = ""
        if lag == 1 and ac_pred > 0.5:
            interp = "⚠ pred very sticky (lagging?)"
        elif lag == 1 and ac_pred < 0.1:
            interp = "✓ pred responsive"

        print(f"  {lag:<6} {ac_actual:>12.4f} {ac_pred:>12.4f} {interp}")

    # -------------------------------------------------------
    # TEST 4: Prediction variation vs actual variation
    # -------------------------------------------------------
    print(f"\n  --- Prediction Dynamics ---")
    print(f"  Actual return std:      {np.std(targets):.6f}")
    print(f"  Predicted return std:   {np.std(preds):.6f}")
    print(f"  Ratio (pred/actual):    {np.std(preds)/np.std(targets):.2f}x")
    print(f"  Actual mean |change|:   {np.mean(np.abs(np.diff(targets))):.6f}")
    print(f"  Pred mean |change|:     {np.mean(np.abs(np.diff(preds))):.6f}")
    print(f"  Ratio (pred/actual):    {np.mean(np.abs(np.diff(preds)))/np.mean(np.abs(np.diff(targets))):.2f}x")

    # If pred changes much less than actual, it's being sluggish/sticky
    pred_change_ratio = np.mean(np.abs(np.diff(preds))) / np.mean(np.abs(np.diff(targets)))
    if pred_change_ratio < 0.3:
        print(f"  ⚠ Predictions are very sluggish — changing {pred_change_ratio:.0%} as fast as actual")
    elif pred_change_ratio < 0.7:
        print(f"  ~ Predictions are somewhat smoothed")
    else:
        print(f"  ✓ Predictions have reasonable dynamics")

    # -------------------------------------------------------
    # TEST 5: Conditional accuracy - new trend vs continuation
    # -------------------------------------------------------
    print(f"\n  --- Accuracy: Trend Start vs Continuation ---")

    # "New trend" = actual direction different from previous bar
    new_trend = np.diff(actual_dir) != 0  # length N-1
    continuation = np.diff(actual_dir) == 0

    # Accuracy on each
    correct = (pred_dir[1:] == actual_dir[1:])  # align with diff

    da_new = np.mean(correct[new_trend]) if new_trend.sum() > 0 else 0
    da_cont = np.mean(correct[continuation]) if continuation.sum() > 0 else 0

    print(f"  Direction reversals (new trend): DA={da_new:.4f} (N={new_trend.sum()})")
    print(f"  Continuations (same direction):  DA={da_cont:.4f} (N={continuation.sum()})")

    if da_cont > da_new + 0.03:
        print(f"  ⚠ Model much better at continuation → likely lagging/copying")
    elif da_new > da_cont + 0.03:
        print(f"  ✓ Model good at detecting reversals → genuine prediction")
    else:
        print(f"  ~ Similar accuracy for both → no strong lag signal")

    return {
        'peak_corr_lag': max_lag,
        'peak_corr': max_corr,
        'da_new_trend': da_new,
        'da_continuation': da_cont,
        'pred_change_ratio': pred_change_ratio,
    }


# ============================================================
# MAIN: Run for 3 stocks
# ============================================================

def run_lag_diagnostic(stocks=['AAPL', 'MSFT', 'SPY']):
    all_results = {}

    for stock in stocks:
        print(f"\n{'#'*70}")
        print(f"# {stock}")
        print(f"{'#'*70}")

        # Load data
        features_df, returns, feature_names = load_and_prepare_data(
            stock, include_vol=True, include_cnn=True
        )
        n_features = len(feature_names)

        # Split
        (train_f, train_r), (val_f, val_r), (test_f, test_r) = split_data(
            features_df, returns, cfg.TRAIN_RATIO, cfg.VAL_RATIO
        )

        # Datasets
        train_ds = ReturnPredictionDataset(train_f, train_r, cfg.SEQ_LEN)
        val_ds = ReturnPredictionDataset(val_f, val_r, cfg.SEQ_LEN)
        test_ds = ReturnPredictionDataset(test_f, test_r, cfg.SEQ_LEN)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

        # Train Step 4
        mode = 'multi_task'
        model = LSTMReturnPredictor(
            n_features=n_features,
            hidden_size=cfg.HIDDEN_SIZE,
            num_layers=cfg.NUM_LAYERS,
            dropout=cfg.DROPOUT,
            mode=mode
        ).to(cfg.DEVICE)

        loss_fn = MultiTaskLoss(a=cfg.GMADL_A, b=cfg.GMADL_B, lambda_cls=cfg.LAMBDA_CLS)

        print(f"  Training Step 4...")
        train_model(model, train_loader, val_loader, loss_fn,
                    mode=mode, epochs=cfg.EPOCHS, patience=cfg.PATIENCE, lr=cfg.LR)

        # Get predictions
        preds, targets = get_predictions(model, test_loader, mode, cfg.DEVICE)

        # Run diagnostic
        result = lag_diagnostic(preds, targets, stock)
        all_results[stock] = result

    # Summary
    print(f"\n\n{'='*70}")
    print(f"  LAG DIAGNOSTIC SUMMARY")
    print(f"{'='*70}")
    print(f"  {'Stock':<8} {'Peak Lag':>10} {'Peak Corr':>10} {'DA(new)':>10} {'DA(cont)':>10} {'Pred/Act Δ':>10} {'Verdict':<20}")
    print(f"  {'-'*8} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*20}")

    for stock, r in all_results.items():
        if r['peak_corr_lag'] == 0:
            verdict = "✓ No lag"
        elif r['peak_corr_lag'] > 0:
            verdict = f"⚠ Lags {r['peak_corr_lag']} bar(s)"
        else:
            verdict = "? Leads"

        print(f"  {stock:<8} {r['peak_corr_lag']:>10} {r['peak_corr']:>10.4f} "
              f"{r['da_new_trend']:>10.4f} {r['da_continuation']:>10.4f} "
              f"{r['pred_change_ratio']:>10.2f}x {verdict:<20}")

    return all_results


run_lag_diagnostic()


######################################################################
# AAPL
######################################################################
  Loaded AAPL: 39289 bars (15min)
  Running Vol model inference...
  Running CNN model inference...
  Training Step 4...
  Epoch   1 | Train: 0.208544 | Val: 0.207702 | DA: 0.4963 | |pred|: 0.930675 | std(pred): 0.537044
  Epoch   5 | Train: 0.207652 | Val: 0.207870 | DA: 0.4989 | |pred|: 2.582414 | std(pred): 1.832525
  Epoch  10 | Train: 0.206824 | Val: 0.207928 | DA: 0.5162 | |pred|: 2.049894 | std(pred): 2.564273
  Epoch  15 | Train: 0.206275 | Val: 0.207667 | DA: 0.5171 | |pred|: 2.788348 | std(pred): 3.388334
  Epoch  20 | Train: 0.205927 | Val: 0.207514 | DA: 0.5111 | |pred|: 3.285266 | std(pred): 3.849237
  Epoch  25 | Train: 0.205320 | Val: 0.207614 | DA: 0.5201 | |pred|: 3.525303 | std(pred): 4.175763
  Epoch  30 | Train: 0.205108 | Val: 0.207839 | DA: 0.5236 | |pred|: 3.751776 | std(pred): 4.393481
  Early stopping at epoch 30


{'AAPL': {'peak_corr_lag': 3,
  'peak_corr': np.float32(0.18989347),
  'da_new_trend': np.float64(0.49750083305564813),
  'da_continuation': np.float64(0.5388418079096046),
  'pred_change_ratio': np.float32(359.6952)},
 'MSFT': {'peak_corr_lag': 1,
  'peak_corr': np.float32(0.09929599),
  'da_new_trend': np.float64(0.49869876382563433),
  'da_continuation': np.float64(0.5573411249086925),
  'pred_change_ratio': np.float32(313.4027)},
 'SPY': {'peak_corr_lag': 1,
  'peak_corr': np.float32(0.064840935),
  'da_new_trend': np.float64(0.4952477936184657),
  'da_continuation': np.float64(0.5241235682054842),
  'pred_change_ratio': np.float32(466.28094)}}

In [ ]:
"""
Lag Diagnostic for Attn-LSTM Turning Point & CNN Dual
=======================================================
Different from GMADL LSTM lag test because:
- Attn-LSTM TP: binary classification (turning point direction)
- CNN Dual: reversal probability (continuous 0-1)

Key questions:
1. Attn-LSTM: Does it predict turning points BEFORE or AFTER they happen?
2. CNN: Does reversal probability peak BEFORE or AFTER the actual reversal?
3. Vol LSTM: Does predicted vol change BEFORE or AFTER actual vol change?

Run after GMADL cell + Train_Save_VolCNN cell are executed.
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# TURNING POINT LAG DIAGNOSTIC
# ============================================================

def tp_lag_diagnostic(df, stock_name):
    """
    Test Attention-LSTM Turning Point model for lag.

    Uses the zigzag turning point labels and model predictions.
    Checks if model predicts direction changes before or after they occur.
    """
    print(f"\n{'='*70}")
    print(f"  TURNING POINT LAG DIAGNOSTIC: {stock_name}")
    print(f"{'='*70}")

    close = df['close'].values

    # --- Generate zigzag turning points (same as training) ---
    REVERSAL_THRESHOLD = 0.01  # 1.0% (best from your experiments)
    MIN_DURATION = 60  # 60 bars minimum
    PRED_HORIZON = 45  # 45-bar prediction horizon

    # Zigzag detection
    turning_points = detect_zigzag(close, REVERSAL_THRESHOLD, MIN_DURATION)

    if len(turning_points) < 10:
        print(f"  Only {len(turning_points)} turning points found, skipping")
        return None

    print(f"  Found {len(turning_points)} turning points")

    # --- Check timing of turning points ---
    # For each TP, look at price action AROUND it
    # If the model is lagging, it would only detect the TP after price has already moved

    tp_indices = [tp[0] for tp in turning_points]
    tp_types = [tp[1] for tp in turning_points]  # 'peak' or 'trough'

    # Analyze: at each turning point, what has price done in the window around it?
    print(f"\n  --- Turning Point Price Action Analysis ---")
    print(f"  (Checking if TP labels are placed at the actual extremum)")

    window = 20  # bars before/after TP to analyze

    early_count = 0  # TP correctly at extremum
    late_count = 0   # actual extremum was earlier

    for i, (tp_idx, tp_type) in enumerate(zip(tp_indices, tp_types)):
        if tp_idx < window or tp_idx >= len(close) - window:
            continue

        local_window = close[tp_idx - window: tp_idx + window + 1]

        if tp_type == 'peak':
            actual_peak_offset = np.argmax(local_window) - window
        else:
            actual_peak_offset = np.argmin(local_window) - window

        if actual_peak_offset == 0:
            early_count += 1  # TP is exactly at extremum
        elif actual_peak_offset < 0:
            late_count += 1   # actual extremum was BEFORE the TP label
        else:
            early_count += 1  # actual extremum was AFTER (TP label is early/ok)

    total = early_count + late_count
    if total > 0:
        print(f"  TP at actual extremum: {early_count}/{total} ({early_count/total*100:.1f}%)")
        print(f"  TP placed late (after extremum): {late_count}/{total} ({late_count/total*100:.1f}%)")

    # --- Build prediction labels (same as training) ---
    # Label: at each bar, what direction is the NEXT turning point?
    labels = np.full(len(close), np.nan)
    for i in range(len(tp_indices) - 1):
        start = tp_indices[i]
        end = tp_indices[i + 1]
        next_tp_type = tp_types[i + 1]
        # If next TP is a peak, price will go UP → label = 1
        # If next TP is a trough, price will go DOWN → label = 0
        direction = 1 if next_tp_type == 'peak' else 0
        labels[start:end] = direction

    valid_mask = ~np.isnan(labels)
    print(f"  Bars with valid TP labels: {valid_mask.sum()}")

    # --- Simple prediction: use momentum as proxy ---
    # In your Attn-LSTM, prediction is based on past 45 bars
    # A lagging model would essentially predict based on recent price direction

    # Check: does recent momentum predict TP direction?
    ret_1 = np.zeros(len(close))
    ret_1[1:] = np.diff(close) / close[:-1]

    momentum_windows = [1, 5, 10, 20, 45]

    print(f"\n  --- Can Simple Momentum Predict TP Direction? ---")
    print(f"  (If yes, Attn-LSTM might just be learning momentum → lag risk)")
    print(f"  {'Momentum Window':<20} {'DA':>7} {'p-value':>10} {'Interpretation':<30}")
    print(f"  {'-'*20} {'-'*7} {'-'*10} {'-'*30}")

    for w in momentum_windows:
        mom = pd.Series(close).pct_change(w).values

        # Predict: if recent momentum is positive → predict UP (label=1)
        pred_from_mom = (mom > 0).astype(float)

        mask = valid_mask & ~np.isnan(mom)
        n = mask.sum()
        if n < 100:
            continue

        correct = (pred_from_mom[mask] == labels[mask])
        da = np.mean(correct)
        n_correct = int(correct.sum())
        p_val = stats.binomtest(n_correct, int(n), 0.5, alternative='greater').pvalue

        # Interpretation
        if da > 0.55:
            interp = "⚠ Momentum predicts TP well → lag risk"
        elif da > 0.52:
            interp = "~ Weak momentum signal"
        elif da < 0.48:
            interp = "✓ Counter-momentum → genuine reversal"
        else:
            interp = "~ No clear pattern"

        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {w:<20} {da:>7.4f} {p_val:>10.6f} {sig:>3} {interp}")

    # --- Check: Attn-LSTM 53.2% DA - is it from momentum or genuine? ---
    # The key test: at ACTUAL turning points, does the model predict
    # the NEW direction, or is it still stuck on the OLD direction?

    print(f"\n  --- Direction at Turning Points ---")
    print(f"  At each TP, what would momentum-based prediction say?")

    for w in [5, 10, 20, 45]:
        mom = pd.Series(close).pct_change(w).values

        correct_at_tp = 0
        wrong_at_tp = 0

        for tp_idx, tp_type in zip(tp_indices, tp_types):
            if tp_idx < w or tp_idx >= len(close) - 1:
                continue

            # At a peak: next direction is DOWN (0)
            # At a trough: next direction is UP (1)
            true_next_dir = 0 if tp_type == 'peak' else 1

            # Momentum prediction at this point
            mom_pred = 1 if mom[tp_idx] > 0 else 0

            if mom_pred == true_next_dir:
                correct_at_tp += 1
            else:
                wrong_at_tp += 1

        total_tp = correct_at_tp + wrong_at_tp
        if total_tp > 0:
            da_tp = correct_at_tp / total_tp
            # At turning points, momentum should be WRONG (it's predicting old direction)
            print(f"  Mom({w:>2}): DA={da_tp:.4f} at TPs (N={total_tp}) "
                  f"{'← momentum WRONG at TPs ✓' if da_tp < 0.45 else '← momentum still works ⚠' if da_tp > 0.55 else ''}")

    return {'n_turning_points': len(turning_points)}


def detect_zigzag(close, threshold, min_duration):
    """Simple zigzag turning point detection."""
    n = len(close)
    if n < 2:
        return []

    turning_points = []
    last_extreme_idx = 0
    last_extreme_val = close[0]
    last_type = None  # 'peak' or 'trough'

    for i in range(1, n):
        if last_type is None:
            # Initialize
            if close[i] > last_extreme_val * (1 + threshold):
                last_type = 'trough'
                turning_points.append((last_extreme_idx, 'trough'))
                last_extreme_idx = i
                last_extreme_val = close[i]
            elif close[i] < last_extreme_val * (1 - threshold):
                last_type = 'peak'
                turning_points.append((last_extreme_idx, 'peak'))
                last_extreme_idx = i
                last_extreme_val = close[i]
        elif last_type == 'trough':
            # Looking for peak
            if close[i] > last_extreme_val:
                last_extreme_idx = i
                last_extreme_val = close[i]
            elif close[i] < last_extreme_val * (1 - threshold):
                if i - last_extreme_idx >= min_duration:
                    turning_points.append((last_extreme_idx, 'peak'))
                    last_type = 'peak'  # should be trough search now
                    # Actually flip: after a peak, search for trough
                    last_type = 'peak'
                    last_extreme_idx = i
                    last_extreme_val = close[i]
        elif last_type == 'peak':
            # Looking for trough
            if close[i] < last_extreme_val:
                last_extreme_idx = i
                last_extreme_val = close[i]
            elif close[i] > last_extreme_val * (1 + threshold):
                if i - last_extreme_idx >= min_duration:
                    turning_points.append((last_extreme_idx, 'trough'))
                    last_type = 'trough'
                    last_extreme_idx = i
                    last_extreme_val = close[i]

    return turning_points


# ============================================================
# CNN REVERSAL LAG DIAGNOSTIC
# ============================================================

@torch.no_grad()
def cnn_lag_diagnostic(df, cnn_model_path, stock_name, device='cpu'):
    """
    Test CNN Dual model for lag.

    Check if CNN reversal probability peaks BEFORE or AFTER actual reversals.
    """
    print(f"\n{'='*70}")
    print(f"  CNN REVERSAL LAG DIAGNOSTIC: {stock_name}")
    print(f"{'='*70}")

    # --- Run CNN inference directly (not via run_cnn_inference, to get raw logits with NaN preserved) ---
    cnn_inputs = compute_cnn_inputs(df, cfg.CNN_WINDOW, cfg.CNN_N_FEATURES)
    cnn_valid = cnn_inputs.dropna()

    print(f"  CNN valid input bars: {len(cnn_valid)} / {len(df)}")

    if not os.path.exists(cnn_model_path):
        print(f"  CNN model not found at {cnn_model_path}")
        return None

    model = CNNDualModel(n_features=cfg.CNN_N_FEATURES, window=cfg.CNN_WINDOW)
    state = torch.load(cnn_model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    model.to(device)

    # Normalize
    cnn_vals = cnn_valid.values.astype(np.float32)
    cnn_mean = cnn_vals.mean(axis=0)
    cnn_std = cnn_vals.std(axis=0) + 1e-8
    cnn_normed = (cnn_vals - cnn_mean) / cnn_std

    # Predict — keep NaN where no prediction
    window = cfg.CNN_WINDOW
    raw_logit_bottom = np.full(len(df), np.nan)
    raw_logit_top = np.full(len(df), np.nan)
    valid_indices = cnn_valid.index.values

    batch_size = 512
    n_predicted = 0
    n_nan_output = 0

    for start in range(window, len(cnn_normed), batch_size):
        end = min(start + batch_size, len(cnn_normed))
        batch_windows = []
        batch_indices = []
        for i in range(start, end):
            w = cnn_normed[i - window:i]
            batch_windows.append(w)
            batch_indices.append(valid_indices[i])

        try:
            x = torch.FloatTensor(np.array(batch_windows)).unsqueeze(1).to(device)
            pb, pt = model(x)

            for j, idx in enumerate(batch_indices):
                pb_val = pb[j].item()
                pt_val = pt[j].item()
                if idx < len(raw_logit_bottom):
                    raw_logit_bottom[idx] = pb_val
                    raw_logit_top[idx] = pt_val
                    if np.isnan(pb_val):
                        n_nan_output += 1
                    else:
                        n_predicted += 1
        except Exception as e:
            print(f"  DEBUG ERROR at start={start}: {e}")
            break

    print(f"  CNN predictions (valid): {n_predicted}, NaN outputs: {n_nan_output}")

    # If all outputs are NaN, try without normalization
    if n_predicted == 0 and n_nan_output > 0:
        print(f"  ⚠ All CNN outputs are NaN! Retrying without normalization...")
        raw_logit_bottom = np.full(len(df), np.nan)
        raw_logit_top = np.full(len(df), np.nan)
        n_predicted = 0

        for start in range(window, len(cnn_vals), batch_size):
            end = min(start + batch_size, len(cnn_vals))
            batch_windows = []
            batch_indices = []
            for i in range(start, end):
                w = cnn_vals[i - window:i]  # unnormalized
                batch_windows.append(w)
                batch_indices.append(valid_indices[i])

            try:
                x = torch.FloatTensor(np.array(batch_windows)).unsqueeze(1).to(device)
                pb, pt = model(x)

                for j, idx in enumerate(batch_indices):
                    pb_val = pb[j].item()
                    pt_val = pt[j].item()
                    if idx < len(raw_logit_bottom) and not np.isnan(pb_val):
                        raw_logit_bottom[idx] = pb_val
                        raw_logit_top[idx] = pt_val
                        n_predicted += 1
            except Exception as e:
                print(f"  DEBUG ERROR (unnorm) at start={start}: {e}")
                break

        print(f"  CNN predictions (unnorm): {n_predicted}")

    # If still failing, try with per-window normalization (z-score each window independently)
    if n_predicted == 0:
        print(f"  ⚠ Still NaN! Trying per-window normalization...")
        raw_logit_bottom = np.full(len(df), np.nan)
        raw_logit_top = np.full(len(df), np.nan)
        n_predicted = 0

        for start in range(window, len(cnn_vals), batch_size):
            end = min(start + batch_size, len(cnn_vals))
            batch_windows = []
            batch_indices = []
            for i in range(start, end):
                w = cnn_vals[i - window:i].copy()
                # Per-window z-score normalization
                w_mean = w.mean(axis=0)
                w_std = w.std(axis=0) + 1e-8
                w = (w - w_mean) / w_std
                batch_windows.append(w)
                batch_indices.append(valid_indices[i])

            try:
                x = torch.FloatTensor(np.array(batch_windows)).unsqueeze(1).to(device)
                pb, pt = model(x)

                for j, idx in enumerate(batch_indices):
                    pb_val = pb[j].item()
                    pt_val = pt[j].item()
                    if idx < len(raw_logit_bottom) and not np.isnan(pb_val):
                        raw_logit_bottom[idx] = pb_val
                        raw_logit_top[idx] = pt_val
                        n_predicted += 1
            except Exception as e:
                print(f"  DEBUG ERROR (per-window) at start={start}: {e}")
                break

        print(f"  CNN predictions (per-window norm): {n_predicted}")

    # Convert to probability but KEEP NaN
    prob_bottom = np.where(np.isnan(raw_logit_bottom), np.nan,
                           1 / (1 + np.exp(-raw_logit_bottom)))
    prob_top = np.where(np.isnan(raw_logit_top), np.nan,
                        1 / (1 + np.exp(-raw_logit_top)))

    valid_mask = ~np.isnan(prob_bottom)
    n_valid = valid_mask.sum()
    print(f"  CNN predictions available: {n_valid}")

    if n_valid < 100:
        print(f"  Not enough CNN predictions, skipping")
        return None

    # Stats on CNN output
    pb_valid = prob_bottom[valid_mask]
    pt_valid = prob_top[valid_mask]
    print(f"  prob_bottom: mean={np.mean(pb_valid):.4f}, std={np.std(pb_valid):.4f}, "
          f"min={np.min(pb_valid):.4f}, max={np.max(pb_valid):.4f}")
    print(f"  prob_top:    mean={np.mean(pt_valid):.4f}, std={np.std(pt_valid):.4f}, "
          f"min={np.min(pt_valid):.4f}, max={np.max(pt_valid):.4f}")

    close = df['close'].values

    # Find actual reversals using zigzag
    REVERSAL_THRESHOLD = 0.005  # 0.5% (matches CNN training)
    turning_points = detect_zigzag(close, REVERSAL_THRESHOLD, min_duration=6)

    print(f"  Found {len(turning_points)} reversals (0.5% threshold)")

    if len(turning_points) < 10:
        print(f"  Not enough turning points, skipping")
        return None

    # --- For each actual reversal, find where CNN prob peaks in surrounding window ---
    print(f"\n  --- CNN Bottom Signal Timing at Actual Troughs ---")
    print(f"  (Positive offset = CNN signal AFTER actual trough = LAGGING)")

    search_window = 10  # bars before/after to search for CNN prob peak

    bottom_offsets = []
    top_offsets = []

    for tp_idx, tp_type in turning_points:
        if tp_idx < search_window or tp_idx >= len(close) - search_window:
            continue

        if tp_type == 'trough':
            local_prob = prob_bottom[tp_idx - search_window: tp_idx + search_window + 1]
            # Skip if too many NaN
            if np.sum(~np.isnan(local_prob)) < search_window:
                continue
            # Replace NaN with min for argmax
            local_clean = np.where(np.isnan(local_prob), np.nanmin(local_prob), local_prob)
            peak_offset = np.argmax(local_clean) - search_window
            bottom_offsets.append(peak_offset)

        elif tp_type == 'peak':
            local_prob = prob_top[tp_idx - search_window: tp_idx + search_window + 1]
            if np.sum(~np.isnan(local_prob)) < search_window:
                continue
            local_clean = np.where(np.isnan(local_prob), np.nanmin(local_prob), local_prob)
            peak_offset = np.argmax(local_clean) - search_window
            top_offsets.append(peak_offset)

    if bottom_offsets:
        offsets = np.array(bottom_offsets)
        print(f"  N troughs analyzed: {len(offsets)}")
        print(f"  Mean offset: {offsets.mean():+.1f} bars")
        print(f"  Median offset: {np.median(offsets):+.1f} bars")
        print(f"  % signal BEFORE trough: {(offsets < 0).mean()*100:.1f}%")
        print(f"  % signal AT trough (±1): {(np.abs(offsets) <= 1).mean()*100:.1f}%")
        print(f"  % signal AFTER trough: {(offsets > 0).mean()*100:.1f}%")

        print(f"\n  Offset distribution (bottom signal vs actual trough):")
        for off in range(-search_window, search_window + 1):
            count = (offsets == off).sum()
            pct = count / len(offsets) * 100
            bar = '#' * int(pct * 2)
            label = ""
            if off == 0: label = "← EXACT"
            elif abs(off) == 1: label = "← LEADS" if off < 0 else "← LAGS"
            print(f"    {off:+3d}: {count:>4} ({pct:>5.1f}%) {bar} {label}")
    else:
        print(f"  No valid troughs to analyze")

    if top_offsets:
        offsets = np.array(top_offsets)
        print(f"\n  --- CNN Top Signal Timing at Actual Peaks ---")
        print(f"  N peaks analyzed: {len(offsets)}")
        print(f"  Mean offset: {offsets.mean():+.1f} bars")
        print(f"  Median offset: {np.median(offsets):+.1f} bars")
        print(f"  % signal BEFORE peak: {(offsets < 0).mean()*100:.1f}%")
        print(f"  % signal AT peak (±1): {(np.abs(offsets) <= 1).mean()*100:.1f}%")
        print(f"  % signal AFTER peak: {(offsets > 0).mean()*100:.1f}%")

        print(f"\n  Offset distribution (top signal vs actual peak):")
        for off in range(-search_window, search_window + 1):
            count = (offsets == off).sum()
            pct = count / len(offsets) * 100
            bar = '#' * int(pct * 2)
            label = ""
            if off == 0: label = "← EXACT"
            elif abs(off) == 1: label = "← LEADS" if off < 0 else "← LAGS"
            print(f"    {off:+3d}: {count:>4} ({pct:>5.1f}%) {bar} {label}")
    else:
        print(f"  No valid peaks to analyze")

    # --- Cross-correlation: prob_bottom vs future returns ---
    print(f"\n  --- Cross-Correlation: CNN prob vs future return ---")

    ret_1 = np.zeros(len(close))
    ret_1[1:] = np.diff(close) / close[:-1]

    # Use only valid (non-NaN) positions
    both_valid = valid_mask & ~np.isnan(ret_1)
    pb_for_corr = prob_bottom[both_valid]
    ret_for_corr = ret_1[both_valid]

    print(f"  Valid samples for correlation: {len(pb_for_corr)}")

    if len(pb_for_corr) > 200:
        print(f"  CNN prob_bottom[t] vs return[t+k]:")
        print(f"  {'k':<6} {'Correlation':>12} {'p-value':>10} {'Interpretation':<30}")

        max_abs_corr = 0
        max_corr_k = 0

        for k in range(-5, 11):
            if k >= 0:
                pb = pb_for_corr[:len(pb_for_corr)-k] if k > 0 else pb_for_corr
                ret = ret_for_corr[k:] if k > 0 else ret_for_corr
            else:
                pb = pb_for_corr[-k:]
                ret = ret_for_corr[:len(ret_for_corr)+k]

            n = min(len(pb), len(ret))
            if n < 100:
                continue

            corr, p_val = stats.pearsonr(pb[:n], ret[:n])

            if abs(corr) > max_abs_corr:
                max_abs_corr = abs(corr)
                max_corr_k = k

            interp = ""
            if k == 0:
                interp = "← prob vs current return"
            elif k > 0:
                interp = f"← prob PREDICTS return {k} bars ahead"
            elif k < 0:
                interp = f"← prob REACTS to return {-k} bars ago"

            marker = " ***" if k == max_corr_k else ""
            print(f"  k={k:<4} {corr:>12.6f} {p_val:>10.6f} {interp}{marker}")

        if max_corr_k > 0:
            print(f"\n  ✓ CNN prob_bottom PREDICTS future returns (peak at k={max_corr_k})")
        elif max_corr_k == 0:
            print(f"\n  ~ CNN prob_bottom is contemporaneous")
        else:
            print(f"\n  ⚠ CNN prob_bottom REACTS to past returns (peak at k={max_corr_k})")

    # --- CNN Autocorrelation ---
    print(f"\n  --- CNN Prediction Autocorrelation ---")
    for lag in [1, 2, 3, 5, 10]:
        if len(pb_for_corr) > lag + 100:
            ac, _ = stats.pearsonr(pb_for_corr[lag:], pb_for_corr[:-lag])
            warning = "⚠ very sticky" if ac > 0.9 else "~ moderate" if ac > 0.5 else "✓ responsive"
            print(f"  AC({lag}): {ac:.4f}  {warning}")

    return {
        'bottom_mean_offset': np.mean(bottom_offsets) if bottom_offsets else None,
        'top_mean_offset': np.mean(top_offsets) if top_offsets else None,
        'n_valid_predictions': n_valid,
    }


# ============================================================
# VOL LSTM LAG DIAGNOSTIC
# ============================================================

@torch.no_grad()
def vol_lag_diagnostic(df, vol_model_path, stock_name, device='cpu'):
    """
    Quick lag check for Vol LSTM.
    Since it predicts block-level (6hr), lag is less critical,
    but still worth checking.
    """
    print(f"\n{'='*70}")
    print(f"  VOL LSTM LAG DIAGNOSTIC: {stock_name}")
    print(f"{'='*70}")

    vol_features = run_vol_inference(
        df, vol_model_path, cfg.VOL_BLOCK_SIZE, cfg.VOL_SEQ_LEN, device
    )

    pred_rv = vol_features['vol_pred_rv'].values
    current_rv = vol_features['vol_current_rv'].values
    pred_dir = vol_features['vol_pred_direction'].values

    valid = ~np.isnan(pred_rv) & ~np.isnan(current_rv)

    print(f"  Valid predictions: {valid.sum()}")

    # Autocorrelation of vol predictions
    pv = pred_rv[valid]
    print(f"\n  --- Vol Prediction Autocorrelation ---")
    for lag in [1, 2, 5, 10, 24]:
        if len(pv) > lag + 100:
            ac = np.corrcoef(pv[lag:], pv[:-lag])[0, 1]
            warning = "⚠ very sticky" if ac > 0.95 else "~ expected for vol" if ac > 0.8 else "✓ responsive"
            print(f"  AC({lag:>2}): {ac:.4f}  {warning}")

    # Check: does predicted vol LEAD or LAG actual vol?
    close = df['close'].values
    ret_1 = np.zeros(len(close))
    ret_1[1:] = np.diff(close) / close[:-1]

    # Actual realized vol (rolling 24-bar)
    actual_rv = pd.Series(ret_1).rolling(24).std().values

    print(f"\n  --- Cross-Correlation: pred_vol[t] vs actual_vol[t+k] ---")
    pv_clean = pred_rv.copy()
    av_clean = actual_rv.copy()
    both_valid = ~np.isnan(pv_clean) & ~np.isnan(av_clean)
    pv_v = pv_clean[both_valid]
    av_v = av_clean[both_valid]

    max_corr = -1
    max_lag = 0

    for k in range(-10, 11):
        if k >= 0:
            p = pv_v[:len(pv_v)-k] if k > 0 else pv_v
            a = av_v[k:] if k > 0 else av_v
        else:
            p = pv_v[-k:]
            a = av_v[:len(av_v)+k]

        n = min(len(p), len(a))
        if n < 100:
            continue

        corr, _ = stats.pearsonr(p[:n], a[:n])

        if corr > max_corr:
            max_corr = corr
            max_lag = k

        if k in [-5, -2, -1, 0, 1, 2, 5, 10]:
            interp = "← CURRENT" if k == 0 else f"← {'predicts' if k > 0 else 'reacts to'} {abs(k)} bars"
            print(f"  k={k:<4} corr={corr:.4f} {interp}")

    if max_lag > 0:
        print(f"\n  ✓ Vol prediction LEADS actual vol by ~{max_lag} bars")
    elif max_lag == 0:
        print(f"\n  ~ Vol prediction contemporaneous with actual vol")
    else:
        print(f"\n  ⚠ Vol prediction LAGS actual vol by ~{-max_lag} bars")


# ============================================================
# MAIN
# ============================================================

def run_all_lag_diagnostics(stocks=['AAPL', 'MSFT', 'SPY']):
    """Run lag diagnostics for all 3 model types."""

    for stock in stocks:
        print(f"\n\n{'#'*75}")
        print(f"# {stock}: ALL MODEL LAG DIAGNOSTICS")
        print(f"{'#'*75}")

        # Load data
        data_dir = os.path.join(cfg.DRIVE_BASE, f"{stock}_{cfg.FREQ_RAW}")
        dfs = []
        for split in ['train', 'test']:
            fpath = os.path.join(data_dir, f"{split}.csv")
            if os.path.exists(fpath):
                dfs.append(load_split_csv(fpath))

        if not dfs:
            print(f"  Data not found for {stock}")
            continue

        df = pd.concat(dfs, ignore_index=True)
        df = df.sort_values('timestamp').reset_index(drop=True)
        df = resample_to_15min(df, cfg.RESAMPLE_PERIOD)
        print(f"  Loaded {stock}: {len(df)} bars")

        # Use TEST portion only (last 15%)
        test_start = int(len(df) * 0.85)
        df_test = df.iloc[test_start:].reset_index(drop=True)
        print(f"  Test portion: {len(df_test)} bars")

        # 1. Turning Point lag
        tp_lag_diagnostic(df_test, stock)

        # 2. CNN lag
        cnn_lag_diagnostic(df_test, cfg.CNN_MODEL_PATH, stock, str(cfg.DEVICE))

        # 3. Vol lag
        vol_lag_diagnostic(df_test, cfg.VOL_MODEL_PATH, stock, str(cfg.DEVICE))

    print(f"\n\n{'='*75}")
    print(f"  SUMMARY")
    print(f"{'='*75}")
    print(f"  If CNN and TP signals LEAD or are AT actual reversals → safe to use")
    print(f"  If they LAG → same problem as GMADL LSTM, not useful for trading")


run_all_lag_diagnostics()



###########################################################################
# AAPL: ALL MODEL LAG DIAGNOSTICS
###########################################################################
  Loaded AAPL: 39289 bars
  Test portion: 5894 bars

  TURNING POINT LAG DIAGNOSTIC: AAPL
  Found 40 turning points

  --- Turning Point Price Action Analysis ---
  (Checking if TP labels are placed at the actual extremum)
  TP at actual extremum: 35/39 (89.7%)
  TP placed late (after extremum): 4/39 (10.3%)
  Bars with valid TP labels: 5684

  --- Can Simple Momentum Predict TP Direction? ---
  (If yes, Attn-LSTM might just be learning momentum → lag risk)
  Momentum Window           DA    p-value Interpretation                
  -------------------- ------- ---------- ------------------------------
  1                     0.5432   0.000000 *** ~ Weak momentum signal
  5                     0.5894   0.000000 *** ⚠ Momentum predicts TP well → lag risk
  10                    0.6246   0.000000 *** ⚠ Mo

In [ ]:
"""
CNN Model Debug
================
Check why CNN model outputs NaN for ALL inputs.
Run after GMADL cell is loaded.
"""

import torch
import numpy as np

# 1. Check model weights for NaN
print("=" * 60)
print("  CNN MODEL WEIGHT CHECK")
print("=" * 60)

model = CNNDualModel(n_features=cfg.CNN_N_FEATURES, window=cfg.CNN_WINDOW)
state = torch.load(cfg.CNN_MODEL_PATH, map_location='cpu')

print(f"\n  Model path: {cfg.CNN_MODEL_PATH}")
print(f"\n  Saved state_dict keys:")
for k, v in state.items():
    has_nan = torch.isnan(v).any().item()
    has_inf = torch.isinf(v).any().item()
    print(f"    {k:<40} shape={str(list(v.shape)):<20} nan={has_nan}  inf={has_inf}  "
          f"mean={v.float().mean().item():.6f}  std={v.float().std().item():.6f}")

# Check if keys match
model_keys = set(model.state_dict().keys())
saved_keys = set(state.keys())
print(f"\n  Keys in model but not in saved: {model_keys - saved_keys}")
print(f"  Keys in saved but not in model: {saved_keys - model_keys}")

# Load and test
model.load_state_dict(state)
model.eval()

# 2. Test with random input
print(f"\n{'='*60}")
print(f"  FORWARD PASS TEST")
print(f"{'='*60}")

for desc, x in [
    ("zeros", torch.zeros(1, 1, 30, 16)),
    ("ones", torch.ones(1, 1, 30, 16)),
    ("randn", torch.randn(1, 1, 30, 16)),
    ("small randn", torch.randn(1, 1, 30, 16) * 0.01),
]:
    try:
        pb, pt = model(x)
        print(f"  Input: {desc:<15} → pb={pb.item():.6f}, pt={pt.item():.6f}, "
              f"nan={torch.isnan(pb).item()}")
    except Exception as e:
        print(f"  Input: {desc:<15} → ERROR: {e}")

# 3. Test with actual data
print(f"\n{'='*60}")
print(f"  TEST WITH REAL DATA")
print(f"{'='*60}")

# Load one stock
data_dir = os.path.join(cfg.DRIVE_BASE, f"AAPL_{cfg.FREQ_RAW}")
dfs = []
for split in ['train', 'test']:
    fpath = os.path.join(data_dir, f"{split}.csv")
    if os.path.exists(fpath):
        dfs.append(load_split_csv(fpath))
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df = resample_to_15min(df, cfg.RESAMPLE_PERIOD)

cnn_inputs = compute_cnn_inputs(df, cfg.CNN_WINDOW, cfg.CNN_N_FEATURES)
cnn_valid = cnn_inputs.dropna()
cnn_vals = cnn_valid.values.astype(np.float32)

print(f"  Data shape: {cnn_vals.shape}")
print(f"  Data stats: mean={cnn_vals.mean():.6f}, std={cnn_vals.std():.6f}")
print(f"  Any NaN in data: {np.isnan(cnn_vals).any()}")
print(f"  Any Inf in data: {np.isinf(cnn_vals).any()}")

# Test first window raw (no normalization)
w = cnn_vals[100:130]  # window of 30
x = torch.FloatTensor(w).unsqueeze(0).unsqueeze(0)  # (1, 1, 30, 16)
print(f"\n  Raw window: shape={x.shape}, min={x.min():.6f}, max={x.max():.6f}")
pb, pt = model(x)
print(f"  Output: pb={pb.item():.6f}, pt={pt.item():.6f}")

# Test with global normalization
cnn_mean = cnn_vals.mean(axis=0)
cnn_std = cnn_vals.std(axis=0) + 1e-8
w_normed = (w - cnn_mean) / cnn_std
x_normed = torch.FloatTensor(w_normed).unsqueeze(0).unsqueeze(0)
print(f"\n  Normed window: min={x_normed.min():.4f}, max={x_normed.max():.4f}")
pb, pt = model(x_normed)
print(f"  Output: pb={pb.item():.6f}, pt={pt.item():.6f}")

# 4. Step through model layers
print(f"\n{'='*60}")
print(f"  LAYER-BY-LAYER FORWARD")
print(f"{'='*60}")

x = torch.FloatTensor(w_normed).unsqueeze(0).unsqueeze(0)  # (1, 1, 30, 16)
print(f"  Input: {x.shape}, nan={torch.isnan(x).any()}")

# Try to step through manually
for name, layer in model.named_children():
    try:
        if 'conv' in name.lower() or 'bn' in name.lower() or 'pool' in name.lower():
            x = layer(x)
            print(f"  After {name}: shape={list(x.shape)}, "
                  f"nan={torch.isnan(x).any().item()}, "
                  f"min={x.min().item():.4f}, max={x.max().item():.4f}")
    except Exception as e:
        print(f"  After {name}: ERROR - {e}")
        break

print(f"\n  Done!")

  CNN MODEL WEIGHT CHECK

  Model path: models/cnn_dual.pt

  Saved state_dict keys:
    conv1.weight                             shape=[32, 1, 3, 16]       nan=True  inf=False  mean=nan  std=nan
    conv1.bias                               shape=[32]                 nan=True  inf=False  mean=nan  std=nan
    conv2.weight                             shape=[64, 32, 3, 1]       nan=True  inf=False  mean=nan  std=nan
    conv2.bias                               shape=[64]                 nan=True  inf=False  mean=nan  std=nan
    fc_bottom.weight                         shape=[1, 64]              nan=True  inf=False  mean=nan  std=nan
    fc_bottom.bias                           shape=[1]                  nan=True  inf=False  mean=nan  std=nan
    fc_top.weight                            shape=[1, 64]              nan=True  inf=False  mean=nan  std=nan
    fc_top.bias                              shape=[1]                  nan=True  inf=False  mean=nan  std=nan

  Keys in model but not in